# MODI-SYNTH · Notebook 05 — the line reader, and every round-2 hypothesis, in one run

**Run with: Accelerator `GPU T4 x2`, Internet `On`, input `colab_pack.zip` attached (the same one notebook 04 uses).
Then *Save Version → Save & Run All (Commit)* and come back in about 5 hours.** Nothing needs watching.

## Why this notebook exists
The CPU investigation (`research/root-cause/LAB_NOTEBOOK.md`) found why notebooks 03/04 never learned, and what works:
cut each patch into lines with a **scale-adaptive** line finder, **pretrain on synthetic Modi lines**, fine-tune on the real
lines, decode with a **character language model**. On one CPU that reached validation CER **42.4** (greedy) and **~41**
with the LM, against 73–75 for the earlier GPU runs. This notebook runs that recipe at GPU scale and, around it, every
round-2 hypothesis — each as one job that changes **one thing** relative to the same anchor, so each answers one question.

## What runs (two GPUs, one job per GPU at a time, highest priority first)
| job | question it answers | id |
|---|---|---|
| `PT_A60` → `FT_A0_s0` | **the anchor**: 60k plain synthetic lines → 30 real epochs | anchor |
| `FT_A0_s1`, `FT_A0_s2` | same as the anchor, other seeds → how big is pure run-to-run noise? | seeds |
| `NOPRE` | real lines only, no synthetic, 2× epochs → can more GPU time alone fix it? | H24-GPU |
| `A12`, `A120`, `PRE_LONG` | 12k vs 60k vs 120k distinct synthetic lines; 3× longer pretraining | N10 |
| `B60` | realistic synthetic: no word gaps + one continuous headline | N4 |
| `C60` | synthetic **text** sampled from a character model of the training labels (new word forms) | N12b |
| `BIG` | a 2× wider network (4× parameters) | N11 |
| `AKS` | akshara (syllable) targets instead of characters | N14 |
| `NS` | no spaces in the targets; the LM puts them back afterwards | N5 |
| `H64` | 64-px lines instead of 40-px (small marks) | N7 |
| `MIX` | keep synthetic lines in the mix during fine-tuning | N9 |
| `AUG` | augment the real lines during fine-tuning | N6 |
| `EM1`, `EM2` | re-align the line labels with the model once / twice during fine-tuning | H9, N8 |
| `FT_LONG` | 3× more fine-tuning epochs → is it still improving? | H11 |
| `COMBO` | built **automatically** from whatever beat the anchor, normal length — do the winners add up? | combination |
| `FINAL_s0`, `FINAL_s1` | **the final model**: the winning recipe on the whole synthetic pool, trained as long as the time allows, two seeds (also ensembled) | final |
| final phase | character-LM decoding grid (N3), checkpoint ensemble (N15), space restoration (N5), **test once** | N3, N15 |

N13 (self-training on HHDoc pages) is **deliberately not here**: HHDoc must not be used for training until the overlap
check with MoDeTrans is done (test-leakage risk). N12 runs as N12b because no external Marathi corpus is attached.

## Rules fixed before the run
* **VAL-SEL** (first 150 validation patches, the same ones the CPU experiments used) picks every epoch, the combination,
  LM weights and the final pipeline. **VAL-REP** (the other 157) is only ever reported. **TEST** is scored once, for the
  anchor and for the pipeline VAL-SEL picks.
* A change is called **better/worse** only if the paired-bootstrap 95% CI excludes zero **and** the difference is larger
  than twice the anchor's seed-to-seed spread. Otherwise: inconclusive.

## What was checked before this was handed to you
* The line finder reproduces the CPU line set **bit for bit** (5,901 lines, 0 of 2,043 patches differ).
* The character alphabet equals the one every earlier notebook used; the akshara alphabet round-trips all 2,043 labels.
* The beam-search decoder gives **identical** output to the CPU decoder (54/54 lines) that produced the 48.4 → 40.8 result.
* `job.py` reproduces the CPU validation CER of the two best CPU checkpoints **exactly** (X4b 52.18, X3a 42.44), and its
  training loop learns: started from X4b, two CPU epochs took validation CER 52.2 → 47.7 → 46.4.
* The faster renderer matches the CPU renderer's distortion strength (0.175 vs 0.181 px) at 3× the speed.
* The scheduler was unit-tested (priorities, dependencies, a failed job's dependents skipped, deadline, learned timing).
* The whole notebook — renderer, benchmark, planner, scheduler, every job type including 64-px and the long jobs, the
  combination, the leftover pass, LM decoding, the report — was rehearsed end to end on CPU with a few patches
  (`notebooks/lr05/rehearse_nb05.py`).

## What was NOT checked, so you know
* **GPU speed.** Every budget is sized from a benchmark this notebook runs on your GPU first; if the GPU is slower than
  expected, jobs are shortened equally (the factor is printed; if it is faster they get up to 2x LONGER) and only the
  lowest-priority jobs can be skipped, never the report.
  The scheduler also learns from every finished job how far off its time estimates are and corrects the rest.
* **The CUDA mixed-precision path and the renderer's low CPU priority** (both standard, both impossible to run on a CPU-only PC).
* **That Kaggle's Pillow can shape Modi text** (it needs `raqm`). Cell 10 checks, tries to install it if missing, and
  says loudly if synthetic text had to be drawn without shaping.

## Outputs (all in `/kaggle/working`)
`REPORT.md` (the answer to every question), `summary.csv`, `logs.txt` (everything printed), `results_bundle.zip` (the
small files, for one download), `runs/<job>/` (log, history, predictions, checkpoint, and `inside_alignment.png`: what
the network emits frame by frame under four validation lines), `figures/`.

In [ ]:
# Cell 1 - environment, GPUs, and the time budget (the whole notebook finishes in about 5 hours)
import os, sys, time, json, re, math, glob, shutil, subprocess, zipfile, base64, hashlib, random
T_START = time.time()
LOCAL = os.environ.get('LR05_LOCAL') == '1'      # set ONLY by the CPU rehearsal script; False on Kaggle
BUDGET_MIN = 40 if LOCAL else 300                # start -> REPORT.md (quality first: Omkar approved ~5 h)
FINAL_RESERVE_MIN = 7 if LOCAL else 20           # kept for LM decoding, ensembles, the test score, the report
COMBO_RESERVE_MIN = 6 if LOCAL else 100          # kept for the combination and the two long FINAL models
OUT = '/kaggle/working'
WORK = ('/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp') + '/lr05'      # big scratch files, not saved
CODE, RUNS, FIGS = f'{OUT}/lr05_code', f'{OUT}/runs', f'{OUT}/figures'
for d in (WORK, CODE, RUNS, FIGS):
    os.makedirs(d, exist_ok=True)
DEADLINE_ALL = T_START + BUDGET_MIN * 60
DEADLINE_JOBS = DEADLINE_ALL - FINAL_RESERVE_MIN * 60

_LOGF = open(f'{OUT}/logs.txt', 'a', encoding='utf-8')
def log(*a):
    s = ' '.join(str(x) for x in a)
    print(s, flush=True); _LOGF.write(s + '\n'); _LOGF.flush()
def clock():
    return time.strftime('%H:%M:%S', time.gmtime(time.time() - T_START))

smi = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
GPUS = [int(l.split()[1].rstrip(':')) for l in smi.splitlines() if l.startswith('GPU')]
import torch, platform
log(f'MODI-SYNTH notebook 05 | started {time.strftime("%Y-%m-%d %H:%M:%S")} | budget {BUDGET_MIN} min | rehearsal: {LOCAL}')
log(f'python {platform.python_version()} | torch {torch.__version__} | CUDA {torch.version.cuda} | CPUs {os.cpu_count()}')
log(smi or 'NO GPU FOUND')
try:
    import psutil
    log(f'RAM {psutil.virtual_memory().total / 2**30:.1f} GB | free disk for scratch files: {shutil.disk_usage(WORK).free / 2**30:.0f} GB')
except Exception:
    pass
assert GPUS or LOCAL, 'Turn on the GPU: Settings -> Accelerator -> GPU T4 x2'
if len(GPUS) == 1:
    log('NOTE: only one GPU is visible. Everything still runs, one job at a time; the planner shortens jobs to fit.')

In [ ]:
# Cell 2 - libraries Kaggle does not ship
!pip -q install datasets editdistance sacrebleu rapidfuzz 2>&1 | tail -1
log('libraries installed')

In [ ]:
# Cell 3 - find colab_pack.zip anywhere under /kaggle/input and unpack the metadata (as in notebook 04)
DATA = f'{WORK}/metadata_unzipped'
os.makedirs(DATA, exist_ok=True)
need = ['splits.json', 'modetrans_text.csv', 'charset.json']

def find_metadata():
    for root, _, files in os.walk('/kaggle/input'):
        if all(n in files for n in need):
            return root
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip'):
                zipfile.ZipFile(os.path.join(root, f)).extractall(DATA)
                for r2, _, f2 in os.walk(DATA):
                    if all(n in f2 for n in need):
                        return r2
    return None

META = find_metadata()
assert META, 'Could not find the metadata. Add colab_pack.zip as an input dataset (Add Input -> Upload).'
log('metadata:', META)

In [ ]:
# Cell 4 - MoDeTrans images + our fixed splits and cleaned labels (identical to notebooks 02-04 and the CPU work)
import numpy as np, pandas as pd
from datasets import load_dataset
for attempt in range(4):
    try:
        ds = load_dataset('historyHulk/MoDeTrans', split='train')     # 2,043 rows
        break
    except Exception as e:
        log(f'download problem (attempt {attempt + 1}): {e}'); time.sleep(30)
names = list(ds['filename'])
splits = json.load(open(f'{META}/splits.json', encoding='utf-8'))['modetrans']['assignment']
text = pd.read_csv(f'{META}/modetrans_text.csv', keep_default_na=False).set_index('image_id')['text_train'].to_dict()
cs = json.load(open(f'{META}/charset.json', encoding='utf-8'))
assert len(names) == 2043 and set(names) == set(splits) == set(text), 'dataset / split mismatch'
SPLIT = {s: [n for n in names if splits[n] == s] for s in ('train', 'val', 'test')}
assert not (set(SPLIT['train']) & (set(SPLIT['val']) | set(SPLIT['test']))), 'leakage!'
if LOCAL:      # rehearsal only: a handful of patches so every code path runs in minutes on a CPU
    SPLIT = {'train': SPLIT['train'][:40], 'val': SPLIT['val'][:22], 'test': SPLIT['test'][:10]}
n_sel = 12 if LOCAL else 150
VAL_SEL, VAL_REP = SPLIT['val'][:n_sel], SPLIT['val'][n_sel:]
TEST, TR150 = SPLIT['test'], SPLIT['train'][:10 if LOCAL else 150]
KEEP = set(SPLIT['train']) | set(SPLIT['val']) | set(SPLIT['test'])
log(f"splits: train {len(SPLIT['train'])} | val {len(SPLIT['val'])} = VAL-SEL {len(VAL_SEL)} (picks epochs and settings) "
    f"+ VAL-REP {len(VAL_REP)} (only reported) | test {len(TEST)} (scored once, at the end)")
META_JSON = dict(symbols=cs['symbols'], train_texts=[text[n] for n in SPLIT['train']], text={n: text[n] for n in KEEP},
                 val_sel=VAL_SEL, val_rep=VAL_REP, test=TEST, train150=TR150, patch_info={})
json.dump(META_JSON, open(f'{WORK}/meta.json', 'w', encoding='utf-8'), ensure_ascii=False)

### Cells 5–9 write the code files
They are written to `/kaggle/working/lr05_code/` and are the tested files verbatim: `lr05_lib.py` (line finder,
renderer, model, decoding, metrics), `render_pools.py` (background synthetic-line renderer), `job.py` (one experiment
on one GPU), `lr05_run.py` (experiment matrix, planner, scheduler), `lr05_final.py` (LM decoding, verdicts, report).

In [ ]:
%%writefile /kaggle/working/lr05_code/lr05_lib.py
"""lr05_lib.py - the line-level Modi reader and every round-2 hypothesis, in one tested file.

Provenance - every piece below was first proven on CPU (research/root-cause/LAB_NOTEBOOK.md):
  segment / split_label / crop_line   X0.3 + X2: each patch's own line pitch; exact line count on 20/24
                                      hand-counted patches (notebook 03's finder: 6/24)
  render_line                         X4: Noto Sans Modi (OFL), Devanagari -> Modi by Unicode name,
                                      print-breaking augmentation. Only change: the elastic wobble is
                                      drawn from a coarse grid (same strength, ~10x cheaper)
  CRNN                                X2/X4b: conv 32-64-128-128-128 + BiLSTM 2x128 + CTC, width stride 2
  viterbi_ctc / realign_labels        X3: CTC forced alignment of the full patch label across its lines
  CharLM / beam_decode                X6: character n-gram LM from TRAINING labels only, fused into a
                                      CTC prefix beam search
New here: akshara targets (N14), no-space targets (N5), LM-sampled synthetic text (N12b), a shared
on-disk line store, the GPU training loop, error anatomy (N1/N2), paired bootstrap comparisons.
"""
import os, re, io, json, math, time, random, collections, unicodedata, threading, queue
import numpy as np
import cv2

# ======================================================================= alphabet and targets
TOK = re.compile(r"<FRAC>|.", re.S)
_C = "क-हक़-य़ॸ-ॿ"                    # consonants
_V = "ऺऻा-ौॎॏॕ-ॗॢॣ"  # dependent vowel signs
_M = "ऀ-ः"                                               # candrabindu, anusvara, visarga
_I = "ऄ-औॠॡॲ-ॷ"                      # independent vowels
# one akshara = (consonant [nukta] virama)* consonant [nukta] (virama | vowel signs*) modifiers*
AKSHARA = re.compile(rf"<FRAC>|(?:[{_C}]़?्)*[{_C}]़?(?:्|[{_V}]*)[{_M}]*|[{_I}][{_M}]*|.", re.S)
COMBINING = re.compile(rf"[{_V}{_M}़्]")


class Vocab:
    """CTC output alphabet.
    kind='char'    : 0 = blank, the 86 dataset symbols, <FRAC>, <unk>. Identical to notebooks 02-04 and the
                     CPU experiments, so numbers stay comparable.
    kind='akshara' : the same, plus every multi-character akshara (syllable cluster) seen >= min_count times
                     in the TRAINING labels. Rarer aksharas are spelled character by character, so any text
                     can still be written and every metric is still computed on characters.
    no_space=True  : spaces are removed from the targets (Modi is written without word gaps; N5)."""

    def __init__(self, symbols, kind="char", train_texts=(), min_count=3, no_space=False):
        self.kind, self.no_space, self.min_count = kind, bool(no_space), min_count
        self.itos = ["<blank>"] + list(symbols) + ["<FRAC>", "<unk>"]
        if kind == "akshara":
            cnt = collections.Counter(a for t in train_texts for a in AKSHARA.findall(t))
            self.itos += sorted(a for a, c in cnt.items() if c >= min_count and len(TOK.findall(a)) > 1)
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.unk = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, s):
        if self.no_space:
            s = s.replace(" ", "")
        if self.kind == "char":
            return [self.stoi.get(t, self.unk) for t in TOK.findall(s)]
        out = []
        for a in AKSHARA.findall(s):
            i = self.stoi.get(a)
            if i is not None:
                out.append(i)
            else:
                out += [self.stoi.get(t, self.unk) for t in TOK.findall(a)]
        return out

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids if i > 0)

    def describe(self):
        return dict(kind=self.kind, no_space=self.no_space, size=len(self.itos))


def make_vocab(symbols, train_texts, kind="char", no_space=False):
    return Vocab(symbols, kind, train_texts, 3, no_space)


# ======================================================================= line segmentation (X0.3, X2)
def row_profile(g):
    return (cv2.GaussianBlur(g, (5, 5), 0) < 128).sum(1).astype(np.float64)


def deskew(g, rng=6.0, step=0.5):
    h, w = g.shape; best, ba = -1.0, 0.0
    for a in np.arange(-rng, rng + 1e-9, step):
        r = g if a == 0 else cv2.warpAffine(g, cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0), (w, h),
                                            flags=cv2.INTER_NEAREST, borderValue=255)
        s = float(np.sum(np.diff(row_profile(r)) ** 2))
        if s > best: best, ba = s, a
    if ba == 0: return g, 0.0
    return cv2.warpAffine(g, cv2.getRotationMatrix2D((w / 2, h / 2), ba, 1.0), (w, h),
                          flags=cv2.INTER_LINEAR, borderValue=255), ba


def estimate_pitch(p, lo=24, hi=260):
    """Dominant period of the row profile. Returns (pitch, confidence)."""
    from scipy.signal import find_peaks
    q = p - p.mean()
    ac = np.correlate(q, q, mode="full")[len(q) - 1:]
    if ac[0] <= 0: return None, 0.0
    ac = ac / ac[0]
    hi = min(hi, len(ac) - 1)
    peaks, props = find_peaks(ac[lo:hi], height=0.05)
    if len(peaks) == 0: return None, 0.0
    k = int(np.argmax(props["peak_heights"]))
    return float(peaks[k] + lo), float(props["peak_heights"][k])


def main_band_height(p):
    """Fallback for one-line patches: height of the largest contiguous block of inked rows."""
    on = p > 0.15 * p.max()
    best = cur = 0
    for v in on:
        cur = cur + 1 if v else 0; best = max(best, cur)
    return float(best)


def segment(g):
    from scipy.signal import find_peaks
    g2, ang = deskew(g)
    p = row_profile(g2)
    pitch, conf = estimate_pitch(p)
    if pitch is None or conf < 0.12:                    # no clear rhythm: treat as a single line
        pitch = max(30.0, main_band_height(p)); conf = 0.0
    sm = cv2.GaussianBlur(p.reshape(-1, 1), (1, 0), sigmaX=0, sigmaY=max(2.0, pitch / 8)).ravel()
    inked = np.where(p > 0.02 * p.max())[0]
    top, bot = (int(inked[0]), int(inked[-1]) + 1) if len(inked) else (0, len(p))
    mins, _ = find_peaks(-sm[top:bot], distance=max(4, int(0.65 * pitch)), prominence=0.04 * sm.max())
    cuts = [top] + [int(m + top) for m in mins] + [bot]
    lines = [(a, b) for a, b in zip(cuts[:-1], cuts[1:]) if (b - a) > 0.35 * pitch and p[a:b].sum() > 0.01 * p.sum()]
    return g2, lines, pitch, conf, ang


def split_label(label, weights):
    """Split `label` into len(weights) pieces, proportional to weights, cutting only at spaces."""
    k = len(weights)
    if k == 1:
        return [label]
    spaces = [i for i, ch in enumerate(label) if ch == " "]
    total = len(label); cum = np.cumsum(weights) / np.sum(weights)
    cuts, prev = [], -1
    for j in range(k - 1):
        target = cum[j] * total
        cand = [s for s in spaces if s > prev]
        if not cand:
            break
        s = min(cand, key=lambda x: abs(x - target))
        cuts.append(s); prev = s
    pieces, start = [], 0
    for c in cuts:
        pieces.append(label[start:c].strip()); start = c + 1
    pieces.append(label[start:].strip())
    while len(pieces) < k:                       # fewer word boundaries than lines: leave the extra lines empty
        pieces.append("")
    return pieces


def crop_line(g2, a, b, heights=(40,), maxw=None):
    """Tight ink crop of rows a:b, scaled to each height in `heights` (one resample each from the crop).
    Returns ({H: image}, ink width in source pixels) or (None, 0) for an empty band."""
    band = g2[a:b]
    ink = band < 128
    rows = np.where(ink.any(1))[0]; cols = np.where(ink.any(0))[0]
    if len(rows) == 0 or len(cols) == 0:
        return None, 0
    r0, r1 = max(0, rows[0] - 2), min(band.shape[0], rows[-1] + 3)
    c0, c1 = max(0, cols[0] - 2), min(band.shape[1], cols[-1] + 3)
    crop = band[r0:r1, c0:c1]
    out = {}
    for H in heights:
        w = max(8, int(round(crop.shape[1] * H / crop.shape[0])))
        w = min(w, (maxw or {}).get(H, int(1024 * H / 40)))
        out[H] = cv2.resize(crop, (w, H), interpolation=cv2.INTER_AREA)
    return out, c1 - c0


def patch_to_lines(g, label, heights=(40, 64)):
    """One patch -> its line images (per height), proportional pseudo labels and patch statistics."""
    g2, bands, pitch, conf, ang = segment(g)
    crops = [crop_line(g2, a, b, heights) for a, b in bands]
    crops = [(c, w) for c, w in crops if c is not None]
    pieces = split_label(label, [w for _, w in crops]) if crops else []
    lines = [dict(imgs=c, label=lab, idx=i, n_lines=len(crops)) for i, ((c, w), lab) in enumerate(zip(crops, pieces))]
    ink = float((g2 < 128).mean())
    return lines, dict(n_lines=len(crops), pitch=float(pitch), pitch_conf=float(conf), angle=float(ang), ink=ink)


# ======================================================================= shared line store
class LineWriter:
    """Appends line images of one height to <base>.u8 and keeps the index in memory. publish() writes the
    index to <base>.json atomically: a reader that sees the .json may use every line it lists, even while
    the writer keeps appending (milestones)."""

    def __init__(self, base, H):
        self.base, self.H = base, H
        self.f = open(base + ".u8", "wb"); self.off = 0
        self.widths, self.offs, self.labels, self.info = [], [], [], []

    def add(self, img, label, info=None):
        assert img.ndim == 2 and img.shape[0] == self.H and img.dtype == np.uint8, (img.shape, img.dtype)
        self.f.write(np.ascontiguousarray(img).tobytes())
        self.offs.append(self.off); self.widths.append(int(img.shape[1])); self.off += int(img.size)
        self.labels.append(label); self.info.append(info)

    def n_lines_ok(self, limit):
        return limit is None or len(self.widths) < limit

    def publish(self, final=False, extra=None):
        self.f.flush(); os.fsync(self.f.fileno())
        meta = dict(H=self.H, n=len(self.widths), final=final, widths=self.widths, offs=self.offs,
                    labels=self.labels, info=self.info, **(extra or {}))
        with open(self.base + ".json.tmp", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False)
        os.replace(self.base + ".json.tmp", self.base + ".json")

    def close(self, extra=None):
        self.publish(final=True, extra=extra)
        self.f.close()


class LineStore:
    """Read side of LineWriter: memory-mapped, so every process shares one copy through the page cache."""

    def __init__(self, base, limit=None):
        meta = json.load(open(base + ".json", encoding="utf-8"))
        n = meta["n"] if limit is None else min(meta["n"], int(limit))
        self.base, self.H, self.n = base, meta["H"], n
        self.widths = np.asarray(meta["widths"][:n], np.int64)
        self.offs = np.asarray(meta["offs"][:n], np.int64)
        self.labels = list(meta["labels"][:n])
        self.info = list(meta["info"][:n])
        self.meta = {k: v for k, v in meta.items() if k not in ("widths", "offs", "labels", "info")}
        self.pix = np.memmap(base + ".u8", dtype=np.uint8, mode="r")

    def __len__(self):
        return self.n

    def img(self, i):
        w = int(self.widths[i]); o = int(self.offs[i])
        return np.asarray(self.pix[o:o + self.H * w]).reshape(self.H, w)


def store_ready(base, need=None):
    try:
        meta_n = json.load(open(base + ".json", encoding="utf-8"))["n"]
    except Exception:
        return False
    return need is None or meta_n >= need


# ======================================================================= synthetic Modi lines (X4)
SIMPLIFY = {"ँ": "ं",   # candrabindu -> anusvara
            "ऑ": "ओ",   # candra O -> O
            "ऱ": "र",   # RRA -> RA
            "़": "",          # nukta -> dropped
            "ऽ": "",          # avagraha -> dropped
            "ॅ": "े",   # vowel sign candra E -> E
            "ॉ": "ो"}   # vowel sign candra O -> O


def to_modi(ch):
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return ch
    if name.startswith("DEVANAGARI "):
        try:
            return unicodedata.lookup(name.replace("DEVANAGARI", "MODI", 1))
        except KeyError:
            return None
    return ch


def normalise(s):
    return "".join(SIMPLIFY.get(c, c) for c in s)


def label_chunks(texts, rng, lo=18, hi=70):
    """Cut texts into line-sized chunks (lo-hi characters) at spaces, like the real lines (median 37)."""
    chunks = []
    for t in texts:
        words = t.split()
        i = 0
        while i < len(words):
            target = int(rng.integers(lo, hi + 1)); cur = []
            while i < len(words) and len(" ".join(cur + [words[i]])) <= target:
                cur.append(words[i]); i += 1
            if not cur:
                cur = [words[i]]; i += 1
            chunks.append(" ".join(cur))
    return [c for c in chunks if c.strip()]


class TextSampler:
    """N12b - new text from a character n-gram model of the TRAINING labels (offline stand-in for an
    external Marathi corpus): sampled from the longest seen context, so words recombine and new
    word forms appear (25% of validation words never occur in training)."""

    def __init__(self, texts, order=4):
        self.order = order
        cnt = collections.defaultdict(collections.Counter)
        for t in texts:
            s = "^" * (order - 1) + t + "$"
            for i in range(order - 1, len(s)):
                for k in range(order):                      # contexts of length 0..order-1
                    cnt[s[i - k:i]][s[i]] += 1
        self.table = {}
        for ctx, c in cnt.items():
            chars = list(c.keys()); p = np.array([c[x] for x in chars], np.float64)
            self.table[ctx] = (chars, np.cumsum(p) / p.sum())
        self.bad = re.compile(rf"(^|\s)[{_V}{_M}़्]|््|[{_V}][{_V}]{{2,}}")

    def _next(self, hist, rng):
        for k in range(self.order - 1, -1, -1):
            ctx = hist[len(hist) - k:] if k else ""
            e = self.table.get(ctx)
            if e is not None:
                chars, cdf = e
                return chars[int(np.searchsorted(cdf, rng.random(), side="right").clip(0, len(chars) - 1))]
        return " "

    def sample(self, rng, lo=18, hi=70, tries=20):
        for _ in range(tries):
            target = int(rng.integers(lo, hi + 1))
            hist = "^" * (self.order - 1); out = []
            while len(out) < target + 25:
                ch = self._next(hist, rng)
                if ch in "$^":
                    hist = "^" * (self.order - 1); ch = " "
                out.append(ch); hist = (hist + ch)[-(self.order - 1):]
            s = "".join(out)
            words = s.split()[1:-1] if len(s.split()) > 2 else s.split()   # drop the clipped first/last word
            text, cur = [], 0
            for w in words:
                if cur + len(w) + (1 if text else 0) > target and text:
                    break
                text.append(w); cur += len(w) + (1 if len(text) > 1 else 0)
            t = " ".join(text).strip()
            if t and not self.bad.search(t):
                return t
        return None


_FONTS = {}


def _font(path, size):
    from PIL import ImageFont
    k = (path, size)
    if k not in _FONTS:
        try:
            _FONTS[k] = ImageFont.truetype(path, size, layout_engine=ImageFont.Layout.RAQM)
        except Exception:
            _FONTS[k] = ImageFont.truetype(path, size)        # basic layout if raqm is missing (logged by the caller)
    return _FONTS[k]


def _coarse_field(h, w, spacing, amp, rng):
    gh, gw = max(2, int(h / spacing) + 2), max(2, int(w / spacing) + 2)
    f = rng.uniform(-1, 1, (gh, gw)).astype(np.float32)
    return cv2.resize(f, (w, h), interpolation=cv2.INTER_CUBIC) * amp


def render_line(label, rng, font_path, heights=(40,), no_gaps=False, headline=False):
    """Render one label in Modi glyphs; return {H: image} (one resample per height from the crop) or None.
    The label keeps its spaces even when no_gaps hides them in the image."""
    from PIL import Image, ImageDraw
    shown = label.replace(" ", "") if no_gaps else label
    modi = "".join(m for m in (to_modi(c) for c in shown) if m is not None)
    if not modi.strip():
        return None
    size = int(rng.integers(34, 60))
    font = _font(font_path, size)
    l, t, r, b = font.getbbox(modi)
    W, Hh = r - l + 20, b - t + 20
    img = Image.new("L", (max(8, W), max(8, Hh)), 255)
    ImageDraw.Draw(img).text((10 - l, 10 - t), modi, font=font, fill=0)
    g = np.array(img)
    if headline:                                       # one continuous top stroke, as Modi scribes wrote
        ink = (g < 128).sum(1)
        top = int(np.argmax(ink[: max(1, int(0.55 * g.shape[0]))]))
        thick = max(2, size // 14)
        cols = np.where((g < 128).any(0))[0]
        if len(cols):
            g = g.copy(); g[max(0, top - thick // 2): top + (thick + 1) // 2, max(0, cols[0] - 4): cols[-1] + 5] = 0
    h, w = g.shape
    shear = rng.uniform(-0.25, 0.25)
    M = np.float32([[1, shear, -shear * h / 2], [0, 1, 0]])
    g = cv2.warpAffine(g, M, (w, h), borderValue=255)
    if rng.random() < 0.7:
        k = np.ones((int(rng.integers(1, 4)), int(rng.integers(1, 4))), np.uint8)
        g = cv2.erode(g, k) if rng.random() < 0.7 else cv2.dilate(g, k)      # erode = thicker dark strokes
    if rng.random() < 0.5:                             # elastic wobble (X4: blurred noise, std ~0.16*a px)
        a = rng.uniform(1.5, 4.0); s = rng.uniform(6, 12)
        dx = _coarse_field(h, w, s, 0.35 * a, rng); dy = _coarse_field(h, w, s, 0.35 * a, rng)
        gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
        g = cv2.remap(g, gx + dx, gy + dy, cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < 0.5:
        M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-2.5, 2.5), 1.0)
        g = cv2.warpAffine(g, M, (w, h), borderValue=255)
    if rng.random() < 0.4:
        g = cv2.GaussianBlur(g, (3, 3), 0)
    if rng.random() < 0.35:
        noise = rng.random(g.shape) < rng.uniform(0.002, 0.02)
        g = g.copy(); g[noise] = 0
    g = np.where(g < 140, 0, 255).astype(np.uint8)    # binarised, like most real patches
    ink = g < 128
    rows = np.where(ink.any(1))[0]; cols = np.where(ink.any(0))[0]
    if len(rows) == 0:
        return None
    g = g[max(0, rows[0] - 2):rows[-1] + 3, max(0, cols[0] - 2):cols[-1] + 3]
    out = {}
    for H in heights:
        nw = max(8, int(round(g.shape[1] * H / g.shape[0])))
        out[H] = cv2.resize(g, (min(nw, int(1024 * H / 40)), H), interpolation=cv2.INTER_AREA)
    return out


# ======================================================================= model
class CRNN:
    pass


def build_crnn(n_classes, size="small"):
    """small = the CPU-proven reader (1.07 M parameters with 89 classes); big = twice as wide (N11).
    Height-agnostic: the conv stack halves the height three times and the rest is averaged."""
    import torch.nn as nn

    class _CRNN(nn.Module):
        def __init__(self, n, c, hid):
            super().__init__()
            L, cin = [], 1
            for i, co in enumerate(c):
                L += [nn.Conv2d(cin, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True)]
                if i == 0: L.append(nn.MaxPool2d(2))              # H -> H/2, width / 2
                if i in (1, 2): L.append(nn.MaxPool2d((2, 1)))    # H/2 -> H/4 -> H/8
                cin = co
            self.cnn = nn.Sequential(*L)
            self.rnn = nn.LSTM(cin, hid, 2, bidirectional=True, batch_first=True, dropout=0.2)
            self.fc = nn.Linear(2 * hid, n)

        def forward(self, x):
            f = self.cnn(x).mean(2).permute(0, 2, 1)                 # (B, W/2, C)
            return self.fc(self.rnn(f)[0])                            # (B, T, classes)

    cfg = dict(small=((32, 64, 128, 128, 128), 128), big=((64, 128, 256, 256, 256), 256))[size]
    return _CRNN(n_classes, *cfg)


# ======================================================================= batches
STRIDE = 2


def line_ok(n_targets, width):
    """CTC feasibility rule used since X2: room for every target plus repeats."""
    return 0 < n_targets and n_targets * 1.2 + 2 <= width / STRIDE


def augment_line(img, rng):
    """X2's light augmentation for real lines (N6): small rotation/scale/shear, stroke thickness."""
    h, w = img.shape
    if rng.random() < 0.5:
        M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-2, 2), rng.uniform(0.92, 1.05))
        M[0, 1] += rng.uniform(-0.15, 0.15)
        img = cv2.warpAffine(img, M, (w, h), borderValue=255)
    if rng.random() < 0.4:
        k = np.ones((2, 2), np.uint8)
        img = cv2.erode(img, k) if rng.random() < 0.5 else cv2.dilate(img, k)
    return img


def to_batch(imgs, H, round_to=32):
    W = max(im.shape[1] for im in imgs)
    W = int(math.ceil(W / round_to) * round_to)
    x = np.ones((len(imgs), 1, H, W), np.float32)
    for i, im in enumerate(imgs):
        x[i, 0, :, :im.shape[1]] = im.astype(np.float32) / 127.5 - 1.0      # (x/255 - 0.5)/0.5, white = 1
    lens = np.array([int(math.ceil(im.shape[1] / STRIDE)) for im in imgs], np.int64)
    return x, lens


def batch_plan(widths, idxs, batch, rng, mega=50):
    """Shuffle, then sort by width inside mega-chunks (fast, little padding), then shuffle batch order."""
    idxs = np.array(idxs, np.int64)
    idxs = idxs[rng.permutation(len(idxs))]
    out = []
    for s in range(0, len(idxs), batch * mega):
        ch = idxs[s:s + batch * mega]
        ch = ch[np.argsort(widths[ch], kind="stable")]
        out += [ch[i:i + batch] for i in range(0, len(ch), batch)]
    order = rng.permutation(len(out))
    return [out[i] for i in order]


class Prefetch:
    """Builds batches in a background thread (numpy/cv2 release the GIL) so the GPU is not kept waiting."""

    def __init__(self, gen, depth=6):
        self.q = queue.Queue(depth); self.err = None
        self.t = threading.Thread(target=self._run, args=(gen,), daemon=True); self.t.start()

    def _run(self, gen):
        try:
            for item in gen:
                self.q.put(item)
        except Exception as e:                      # surface worker errors in the main thread
            self.err = e
        self.q.put(None)

    def __iter__(self):
        while True:
            item = self.q.get()
            if item is None:
                if self.err is not None:
                    raise self.err
                return
            yield item


# ======================================================================= decoding and metrics
def greedy_ids(lp_row):
    s = lp_row.argmax(-1)
    keep = np.ones_like(s, bool); keep[1:] = s[1:] != s[:-1]; keep &= s != 0
    return s[keep]


def cer_parts(ref, hyp):
    import editdistance
    u = lambda s: s.replace("<FRAC>", "")
    return editdistance.eval(u(ref), u(hyp)), max(1, len(u(ref)))


def metrics(refs, hyps):
    import editdistance, sacrebleu
    u = lambda s: s.replace("<FRAC>", "")
    ns = lambda s: u(s).replace(" ", "")
    e = np.array([editdistance.eval(u(r), u(h)) for r, h in zip(refs, hyps)], float)
    L = np.array([max(1, len(u(r))) for r in refs], float)
    e2 = sum(editdistance.eval(ns(r), ns(h)) for r, h in zip(refs, hyps)); n2 = sum(len(ns(r)) for r in refs)
    we = sum(editdistance.eval(r.split(), h.split()) for r, h in zip(refs, hyps)); wl = sum(max(1, len(r.split())) for r in refs)
    per = e / L
    return dict(CER=100 * e.sum() / L.sum(), CER_nospace=100 * e2 / max(1, n2), WER=100 * we / max(1, wl),
                BLEU=sacrebleu.corpus_bleu(hyps, [refs]).score,
                chrF_pp=sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score,
                median_patch_CER=100 * float(np.median(per)), patches_below_50=int((per < 0.5).sum()),
                best_patch_CER=100 * float(per.min()), len_ratio=float(np.mean([len(h) for h in hyps]) / max(1e-9, np.mean([len(r) for r in refs]))),
                n=len(refs))


def edit_anatomy(refs, hyps, focus=(" ", "ा", "्", "र", "ं", "ि", "ी", "ु", "ू", "े", "ो")):
    """Deletions / insertions / substitutions, the most-missed characters and the top confusions."""
    from rapidfuzz.distance import Levenshtein
    D = I = S = 0; miss = collections.Counter(); ins = collections.Counter(); sub = collections.Counter()
    ref_cnt = collections.Counter()
    for r, h in zip(refs, hyps):
        r = r.replace("<FRAC>", "\x01"); h = h.replace("<FRAC>", "\x01")
        ref_cnt.update(r)
        for op in Levenshtein.editops(r, h):
            if op.tag == "delete": D += 1; miss[r[op.src_pos]] += 1
            elif op.tag == "insert": I += 1; ins[h[op.dest_pos]] += 1
            else: S += 1; sub[(r[op.src_pos], h[op.dest_pos])] += 1
    tot = max(1, D + I + S)
    recall = {c: round(100 * (1 - (miss[c] + sum(v for (a, b), v in sub.items() if a == c)) / max(1, ref_cnt[c])), 1) for c in focus}
    return dict(errors=D + I + S, deletions_pct=round(100 * D / tot, 1), insertions_pct=round(100 * I / tot, 1),
                substitutions_pct=round(100 * S / tot, 1),
                most_missed=[(c, n) for c, n in miss.most_common(12)],
                most_inserted=[(c, n) for c, n in ins.most_common(8)],
                top_confusions=[(a + "->" + b, n) for (a, b), n in sub.most_common(15)],
                focus_char_recall=recall)


def paired_bootstrap(e_a, e_b, L, n=10000, seed=0):
    """95% CI of CER(a) - CER(b) over patches resampled together (same patches for both systems)."""
    e_a, e_b, L = (np.asarray(v, float) for v in (e_a, e_b, L))
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(L), (n, len(L)))
    d = 100 * (e_a[idx].sum(1) - e_b[idx].sum(1)) / L[idx].sum(1)
    return float(100 * (e_a.sum() - e_b.sum()) / L.sum()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def spearman(a, b):
    import warnings
    from scipy.stats import spearmanr
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 5 or np.ptp(a[ok]) == 0 or np.ptp(b[ok]) == 0:
        return float("nan")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(a[ok], b[ok]).correlation)


# ======================================================================= CTC forced alignment (X3)
def viterbi_ctc(logp, labels):
    """Best CTC path of `labels` through log-probs (T, C). Returns, for every frame, the index of the label
    token it is assigned to (blanks belong to the token before them) and the path score."""
    T = logp.shape[0]; L = len(labels)
    S = 2 * L + 1
    ext = np.zeros(S, dtype=int); ext[1::2] = labels
    NEG = -1e30
    dp = np.full((T, S), NEG); bp = np.zeros((T, S), dtype=np.int8)
    dp[0, 0] = logp[0, 0]
    if S > 1: dp[0, 1] = logp[0, ext[1]]
    allow2 = np.zeros(S, bool)
    allow2[3::2] = ext[3::2] != ext[1:-2:2]
    for t in range(1, T):
        prev = dp[t - 1]
        c0 = prev
        c1 = np.concatenate([[NEG], prev[:-1]])
        c2 = np.where(allow2, np.concatenate([[NEG, NEG], prev[:-2]]), NEG)
        stack = np.stack([c0, c1, c2])
        k = stack.argmax(0)
        dp[t] = stack[k, np.arange(S)] + logp[t, ext]
        bp[t] = k
    s = S - 1 if dp[T - 1, S - 1] >= dp[T - 1, S - 2] else S - 2
    path = np.empty(T, dtype=int)
    for t in range(T - 1, -1, -1):
        path[t] = s
        s -= bp[t, s]
    char_of_frame = np.where(path % 2 == 1, path // 2, -1)
    last = -1
    for t in range(T):
        if char_of_frame[t] >= 0: last = char_of_frame[t]
        else: char_of_frame[t] = last
    return char_of_frame, float(dp[T - 1, path[-1]])


def realign_patch(line_logps, label, vocab):
    """X3 for one patch: force-align the FULL label across its lines (laid end to end) and cut it where the
    alignment crosses into the next line, snapped to the nearest space. Returns the new line labels or None."""
    toks = AKSHARA.findall(label) if vocab.kind == "akshara" else TOK.findall(label)
    ids = []
    for a in toks:                                   # token ids of the label, keeping a map back to the strings
        ids.append(vocab.encode(a))
    flat = [i for x in ids for i in x]
    owner_tok = [k for k, x in enumerate(ids) for _ in x]
    logp = np.concatenate(line_logps); owner = np.concatenate([[i] * lp.shape[0] for i, lp in enumerate(line_logps)])
    if not flat or len(flat) > logp.shape[0]:
        return None
    cof, _ = viterbi_ctc(logp, np.array(flat))
    starts = [0]
    for i in range(1, len(line_logps)):
        ci = cof[owner == i]; ci = ci[ci >= 0]
        starts.append(owner_tok[int(ci.min())] if len(ci) else starts[-1])
    spaces = [j for j, t in enumerate(toks) if t == " "]
    cuts = []
    for st in starts[1:]:
        cand = [s for s in spaces if (not cuts or s > cuts[-1])]
        cuts.append(min(cand, key=lambda s: abs(s + 1 - st)) if cand else (cuts[-1] if cuts else 0))
    pieces, b = [], 0
    for c in cuts:
        pieces.append("".join(toks[b:c]).strip()); b = c + 1
    pieces.append("".join(toks[b:]).strip())
    return pieces


# ======================================================================= character LM and beam search (X6)
class CharLM:
    """Character n-gram LM with interpolated absolute discounting, built from TRAINING labels only."""

    def __init__(self, texts, n, d=0.75):
        self.n, self.d = n, d
        self.c = [collections.Counter() for _ in range(n + 1)]
        self.ctx = [collections.Counter() for _ in range(n + 1)]
        self.types = [collections.defaultdict(set) for _ in range(n + 1)]
        V = set()
        for t in texts:
            s = ["^"] * (n - 1) + TOK.findall(t) + ["$"]
            V.update(s)
            for k in range(1, n + 1):
                for i in range(k - 1, len(s)):
                    h = tuple(s[i - k + 1:i]); w = s[i]
                    self.c[k][h + (w,)] += 1; self.ctx[k][h] += 1; self.types[k][h].add(w)
        self.V = len(V)
        self.cache = {}

    def logp(self, hist, w):
        key = (tuple(hist[-(self.n - 1):]) if self.n > 1 else (), w)
        v = self.cache.get(key)
        if v is not None:
            return v
        hist = list(key[0])
        p = 1.0 / self.V
        for k in range(1, self.n + 1):
            h = tuple(hist[len(hist) - (k - 1):]) if k > 1 else ()
            if len(h) < k - 1: continue
            cn = self.ctx[k][h]
            if cn == 0: continue
            lam = self.d * len(self.types[k][h]) / cn
            p = max(self.c[k][h + (w,)] - self.d, 0) / cn + lam * p
        v = math.log(p)
        self.cache[key] = v
        return v


def topk_compress(lp, k=8):
    """Keep the k most likely tokens per frame (all the beam search ever looks at)."""
    idx = np.argsort(lp, axis=1)[:, -k:][:, ::-1]
    return idx.astype(np.int32), np.take_along_axis(lp, idx, 1).astype(np.float32)


def beam_decode(top_idx, top_lp, itos, lm, alpha, beta, beam=10):
    """CTC prefix beam search with shallow LM fusion over TOKEN STRINGS (a token may be one character or a
    whole akshara; its LM score is the sum over its characters). Identical to X6 for the char alphabet."""
    order = lm.n
    lse = lambda a, b: max(a, b) + math.log1p(math.exp(-abs(a - b))) if min(a, b) > -1e29 else max(a, b)
    toks_of = {}
    tail = {(): ("^",) * (order - 1)}
    beams = {(): (0.0, -1e30)}
    lmc = {}

    def lmscore(prefix, c):
        key = (tail[prefix], c)
        v = lmc.get(key)
        if v is None:
            ts = toks_of.get(c)
            if ts is None:
                ts = toks_of[c] = TOK.findall(itos[c])
            h = list(tail[prefix]); v = 0.0
            for t in ts:
                v += lm.logp(h, t); h.append(t)
            lmc[key] = v
        return v

    for t in range(top_idx.shape[0]):
        nb = collections.defaultdict(lambda: [-1e30, -1e30])
        for prefix, (pb, pnb) in beams.items():
            tot = lse(pb, pnb)
            for c, p in zip(top_idx[t], top_lp[t]):
                c = int(c); p = float(p)
                if c == 0:
                    e = nb[prefix]; e[0] = lse(e[0], tot + p)
                    continue
                last = prefix[-1] if prefix else None
                ext = prefix + (c,)
                if ext not in tail:
                    ts = toks_of.get(c)
                    if ts is None:
                        ts = toks_of[c] = TOK.findall(itos[c])
                    tail[ext] = tuple((list(tail[prefix]) + ts)[-(order - 1):]) if order > 1 else ()
                bonus = alpha * lmscore(prefix, c) + beta
                if c == last:
                    e = nb[ext]; e[1] = lse(e[1], pb + p + bonus)
                    e2 = nb[prefix]; e2[1] = lse(e2[1], pnb + p)
                else:
                    e = nb[ext]; e[1] = lse(e[1], tot + p + bonus)
        beams = dict(sorted(((k, (v[0], v[1])) for k, v in nb.items()), key=lambda kv: -lse(*kv[1]))[:beam])
    best = max(beams.items(), key=lambda kv: lse(*kv[1]))[0]
    return "".join(itos[i] for i in best if i > 0)


def restore_spaces(s, lm, gamma, beam=16):
    """N5: put word spaces back into a space-free reading with the character LM (spaces never go before a
    combining mark). gamma = bonus per inserted space, tuned on validation."""
    toks = TOK.findall(s)
    if not toks:
        return s
    order = lm.n
    beams = [(0.0, ("^",) * (order - 1), [])]
    for i, t in enumerate(toks):
        nb = []
        for sc, tl, out in beams:
            opts = [False]
            if i > 0 and out and out[-1] != " " and not COMBINING.match(t):
                opts.append(True)
            for sp in opts:
                h = list(tl); sc2 = sc; o2 = list(out)
                if sp:
                    sc2 += lm.logp(h, " ") + gamma; h.append(" "); o2.append(" ")
                sc2 += lm.logp(h, t); h.append(t); o2.append(t)
                nb.append((sc2, tuple(h[-(order - 1):]), o2))
        nb.sort(key=lambda x: -x[0]); beams = nb[:beam]
    return "".join(beams[0][2])


# ======================================================================= small helpers
class Tee:
    """print() to the notebook AND to a log file, flushed line by line."""

    def __init__(self, path):
        self.f = open(path, "a", encoding="utf-8")

    def __call__(self, *a):
        s = " ".join(str(x) for x in a)
        print(s, flush=True)
        self.f.write(s + "\n"); self.f.flush()


def fmt_s(sec):
    sec = int(max(0, sec)); return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}"


def gpu_status():
    import subprocess
    try:
        q = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total,temperature.gpu",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout.strip()
        return " | ".join(f"GPU{r.split(',')[0].strip()} {r.split(',')[1].strip()}% {r.split(',')[2].strip()}/{r.split(',')[3].strip()}MB {r.split(',')[4].strip()}C"
                          for r in q.splitlines() if r.strip())
    except Exception:
        return "nvidia-smi unavailable"

In [ ]:
%%writefile /kaggle/working/lr05_code/render_pools.py
"""render_pools.py - render the synthetic line pools in the background while the GPUs train.

Pools are rendered one after another and published at milestones (the index file is rewritten
atomically), so a job that needs only the first 12,000 or 60,000 lines of pool A can start long before
all of pool A exists. Text comes from TRAINING labels only (or from a character n-gram model of them),
so no validation or test text can leak into the synthetic data.
usage: python render_pools.py --work DIR --font TTF --plan plan.json --procs 3
"""
import argparse, json, os, sys, time
from multiprocessing import Pool
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import lr05_lib as L

_G = {}


def _init(font, texts, seed):
    _G.update(font=font, texts=texts, sampler=None)
    _G["chunks"] = L.label_chunks(texts, np.random.default_rng(seed))


def _render(task):
    style, text_src, heights, chunk_id, n, seed = task
    rng = np.random.default_rng([seed, chunk_id])
    if text_src == "lm" and _G["sampler"] is None:
        _G["sampler"] = L.TextSampler(_G["texts"], order=4)
    out, tries = [], 0
    while len(out) < n and tries < 3 * n:
        tries += 1
        lab = _G["sampler"].sample(rng) if text_src == "lm" else _G["chunks"][int(rng.integers(len(_G["chunks"])))]
        if not lab:
            continue
        o = L.render_line(lab, rng, _G["font"], heights, no_gaps=(style == "realistic"), headline=(style == "realistic"))
        if o is not None:
            out.append((o, lab))
    return chunk_id, out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work", required=True); ap.add_argument("--font", required=True)
    ap.add_argument("--plan", required=True); ap.add_argument("--procs", type=int, default=3)
    ap.add_argument("--chunk", type=int, default=200)
    a = ap.parse_args()
    plan = json.load(open(a.plan, encoding="utf-8"))
    meta = json.load(open(os.path.join(a.work, "meta.json"), encoding="utf-8"))
    texts = [L.normalise(t).replace("<FRAC>", "") for t in meta["train_texts"]]
    print(f"renderer: {len(texts)} training labels, {a.procs} processes, pools {[p['name'] for p in plan]}", flush=True)
    t_all = time.time()
    with Pool(a.procs, initializer=_init, initargs=(a.font, texts, 0)) as pool:
        for P in plan:
            t0 = time.time()
            heights = tuple(sorted(set([40] + ([64] if P.get("h64_limit") else []))))
            W = {40: L.LineWriter(os.path.join(a.work, f"pool_{P['name']}_h40"), 40)}
            if 64 in heights:
                W[64] = L.LineWriter(os.path.join(a.work, f"pool_{P['name']}_h64"), 64)
            n_chunks = int(np.ceil(P["n"] / a.chunk))
            tasks = [(P["style"], P["text"], heights, c, min(a.chunk, P["n"] - c * a.chunk), P["seed"]) for c in range(n_chunks)]
            done, next_report, ms = 0, 0.1, sorted(P.get("milestones", []))
            extra = dict(style=P["style"], text=P["text"], seed=P["seed"])
            for cid, lines in pool.imap(_render, tasks):
                for o, lab in lines:
                    W[40].add(o[40], lab)
                    if 64 in W and W[64].n_lines_ok(P["h64_limit"]):
                        W[64].add(o[64], lab)
                    done += 1
                    if ms and done >= ms[0]:
                        W[40].publish(extra=extra)
                        if 64 in W: W[64].publish(extra=extra)
                        print(f"  pool {P['name']}: milestone {ms[0]:,} lines published ({time.time()-t0:.0f}s)", flush=True)
                        ms.pop(0)
                if done >= next_report * P["n"]:
                    r = done / max(1e-9, time.time() - t0)
                    print(f"  pool {P['name']}: {done:,}/{P['n']:,} lines | {r:.0f} lines/s | "
                          f"ETA {L.fmt_s((P['n'] - done) / max(r, 1e-9))}", flush=True)
                    next_report += 0.1
            for w in W.values():
                w.close(extra=extra)
            wid = np.array(W[40].widths); nch = np.array([len(x) for x in W[40].labels])
            print(f"POOL {P['name']} DONE: {len(wid):,} lines ({P['style']}, text={P['text']}) in {time.time()-t0:.0f}s | "
                  f"width@40 median {np.median(wid):.0f} | chars median {np.median(nch):.0f} | "
                  f"frames/char median {np.median(wid / 2 / np.maximum(1, nch)):.1f}", flush=True)
    print(f"RENDERER FINISHED in {time.time()-t_all:.0f}s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lr05_code/job.py
"""job.py - one experiment of notebook 05, in its own process, pinned to one GPU.

A job = [pretrain on a synthetic pool] -> [fine-tune on real lines] -> evaluate. It logs everything to stdout
(streamed into the notebook with a [tag] prefix) and to its run directory: log.txt, history.csv,
result.json, val_sel/val_rep/train150 predictions, best.pt (and pre.pt for shared pretraining).
A crash still writes result.json (status 'failed' + traceback), so the notebook carries on with the rest.
usage: python job.py --spec spec.json [--gpu 0]
"""
import argparse, csv, json, math, os, random, sys, time, traceback

ap = argparse.ArgumentParser()
ap.add_argument("--spec", required=True)
ap.add_argument("--gpu", default=None)
args = ap.parse_args()
if args.gpu is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)          # before torch is imported
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import lr05_lib as L

S = json.load(open(args.spec, encoding="utf-8"))
RUN = S["run_dir"]; os.makedirs(RUN, exist_ok=True)
log = L.Tee(os.path.join(RUN, "log.txt"))
dev = torch.device("cuda" if torch.cuda.is_available() and not S.get("cpu") else "cpu")
AMP = dev.type == "cuda"
torch.backends.cudnn.benchmark = False
if dev.type == "cpu":
    torch.set_num_threads(int(S.get("threads", 2)))
T_JOB = time.time()
DEADLINE = min(float(S.get("deadline", 1e18)), T_JOB + float(S.get("time_cap_s", 1e9)))
EVAL_RESERVE = float(S.get("eval_reserve_s", 75))            # stop training this long before the deadline
RESULT = dict(tag=S["tag"], hyp=S.get("hyp", []), desc=S.get("desc", ""), status="running", capped=False,
              spec={k: v for k, v in S.items() if k not in ("work", "run_dir")})


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=AMP)


def autocast():
    return torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=AMP)


def gpu_mem():
    return f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GB" if dev.type == "cuda" else "cpu"


# ======================================================================= data
W = S["work"]
META = json.load(open(os.path.join(W, "meta.json"), encoding="utf-8"))
TEXT = META["text"]
VAL_SEL, VAL_REP, TR150 = META["val_sel"], META["val_rep"], META["train150"]
PINFO = META["patch_info"]
H = int(S["H"])
vocab = L.make_vocab(META["symbols"], META["train_texts"], S["vocab"], S.get("no_space", False))
real = L.LineStore(os.path.join(W, f"real_h{H}"))
by_patch = {}
for i, inf in enumerate(real.info):
    by_patch.setdefault(inf["patch"], []).append(i)
for p in by_patch:
    by_patch[p].sort(key=lambda i: real.info[i]["idx"])
TRAIN_LINES = [i for i, inf in enumerate(real.info) if inf["split"] == "train"]
TRAIN_PATCHES = sorted({real.info[i]["patch"] for i in TRAIN_LINES})
SHOW = VAL_SEL[0]                                           # the patch whose reading is printed every evaluation


@torch.no_grad()
def predict(model, store, groups, want_lp=False, bs=64):
    """Greedy reading of every line (and optionally its log-probs). Returns (texts, lps, blank share).
    groups = lists of line indices, one list per PATCH: each patch is read as its own batch, padded only to its
    widest line rounded to the stride - exactly how the CPU experiments read, so their numbers reproduce. (A BiLSTM's
    output depends slightly on how much padding follows a line, so batch composition must not vary between runs.)"""
    model.eval()
    out, lps, blank, tot = {}, {}, 0, 0
    for g in groups:
        for s in range(0, len(g), bs):
            b = g[s:s + bs]
            if not b:
                continue
            x, lens = L.to_batch([store.img(i) for i in b], store.H, round_to=L.STRIDE)
            with autocast():
                o = model(torch.from_numpy(x).to(dev))
            lp = o.float().log_softmax(-1).cpu().numpy()
            for k, i in enumerate(b):
                row = lp[k, :lens[k]]
                am = row.argmax(-1); blank += int((am == 0).sum()); tot += len(am)
                out[i] = vocab.decode(L.greedy_ids(row))
                if want_lp:
                    lps[i] = row
    model.train()
    return out, lps, blank / max(1, tot)


def eval_patches(model, patches, save=None):
    pred, _, blank = predict(model, real, [by_patch.get(p, []) for p in patches])
    sep = "" if vocab.no_space else " "
    hyps = [sep.join(t for t in (pred[i].strip() for i in by_patch.get(p, [])) if t) for p in patches]
    refs = [TEXT[p] for p in patches]
    m = L.metrics(refs, hyps)
    m["blank_pct"] = 100 * blank
    m["primary"] = m["CER_nospace"] if vocab.no_space else m["CER"]
    if save:
        import editdistance
        u = lambda s: s.replace("<FRAC>", ""); ns = lambda s: u(s).replace(" ", "")
        with open(os.path.join(RUN, save), "w", encoding="utf-8", newline="") as f:
            wr = csv.writer(f); wr.writerow(["patch", "ref", "hyp", "err", "n", "err_nospace", "n_nospace"])
            for p, r, h in zip(patches, refs, hyps):
                wr.writerow([p, r, h, editdistance.eval(u(r), u(h)), len(u(r)), editdistance.eval(ns(r), ns(h)), len(ns(r))])
    return m, refs, hyps


def fmt_m(m):
    return (f"CER {m['CER']:5.1f} | no-space {m['CER_nospace']:5.1f} | WER {m['WER']:5.1f} | BLEU {m['BLEU']:5.2f} | "
            f"chrF++ {m['chrF_pp']:5.1f} | median patch {m['median_patch_CER']:5.1f} | <50%: {m['patches_below_50']}/{m['n']} | "
            f"len {m['len_ratio']:.2f}x | blank {m['blank_pct']:4.1f}%")


HIST = []


def record(stage, step, samples, loss, lr, m, extra=None):
    row = dict(stage=stage, step=step, samples=samples, loss=round(loss, 4) if loss == loss else None, lr=lr,
               CER=round(m["CER"], 3), CER_nospace=round(m["CER_nospace"], 3), primary=round(m["primary"], 3),
               WER=round(m["WER"], 2), BLEU=round(m["BLEU"], 3), chrF_pp=round(m["chrF_pp"], 2),
               below50=m["patches_below_50"], len_ratio=round(m["len_ratio"], 3), blank_pct=round(m["blank_pct"], 2),
               minutes=round((time.time() - T_JOB) / 60, 2), **(extra or {}))
    HIST.append(row)
    with open(os.path.join(RUN, "history.csv"), "w", encoding="utf-8", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=list(dict.fromkeys(k for r in HIST for k in r)))
        wr.writeheader(); wr.writerows(HIST)


# ======================================================================= training machinery
class Stage:
    """Adam + linear warm-up + cosine decay to 2% of the peak, AMP on GPU, gradient clipping at 5 (as on CPU)."""

    def __init__(self, name, model, lr, total_steps, warmup=200, log_every=100):
        self.name, self.model, self.total = name, model, max(1, int(total_steps))
        self.opt = torch.optim.Adam(model.parameters(), lr=lr)
        wu = min(warmup, max(1, self.total // 10))
        f = lambda s: min(1.0, (s + 1) / wu) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / self.total))))
        self.sched = torch.optim.lr_scheduler.LambdaLR(self.opt, f)
        self.scaler = make_scaler()
        self.step = self.seen = self.skipped = 0
        self.ema = None; self.gmax = 0.0; self.t0 = time.time(); self.win = (time.time(), 0)
        self.log_every = log_every; self.last_log = time.time()

    def update(self, batch):
        x, lens, y, yl = batch
        x = x.to(dev, non_blocking=True)
        with autocast():
            out = self.model(x)
        lp = out.float().log_softmax(-1)
        loss = F.ctc_loss(lp.permute(1, 0, 2), y.to(dev), lens, yl, blank=0, zero_infinity=True)
        self.opt.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            self.skipped += 1; self.step += 1; self.sched.step(); return
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.opt)
        g = float(nn.utils.clip_grad_norm_(self.model.parameters(), 5.0))
        self.scaler.step(self.opt); self.scaler.update(); self.sched.step()
        if not math.isfinite(g):
            self.skipped += 1
        else:
            self.gmax = max(self.gmax, g)
        lv = float(loss.item())
        self.ema = lv if self.ema is None else 0.98 * self.ema + 0.02 * lv
        self.step += 1; self.seen += x.shape[0]
        if self.step % self.log_every == 0 or time.time() - self.last_log > 90:
            am = lp.detach().argmax(-1)
            valid = torch.arange(am.shape[1], device=am.device)[None, :] < lens.to(am.device)[:, None]
            blank = float(((am == 0) & valid).sum()) / max(1.0, float(valid.sum()))
            nz = ((am != 0) & valid); chg = torch.ones_like(am, dtype=torch.bool); chg[:, 1:] = am[:, 1:] != am[:, :-1]
            plen = float((nz & chg).sum()) / max(1, x.shape[0]); tlen = float(yl.float().mean())
            t1, n1 = self.win; rate = (self.seen - n1) / max(1e-9, time.time() - t1); self.win = (time.time(), self.seen)
            el = time.time() - self.t0; eta = el / max(1, self.step) * (self.total - self.step)
            log(f"[{self.name} {self.step:>6}/{self.total} {100*self.step/self.total:3.0f}%] loss {lv:6.3f} (ema {self.ema:6.3f}) | "
                f"lr {self.sched.get_last_lr()[0]:.2e} | |g| {g:6.2f} (max {self.gmax:6.1f}) | blank {100*blank:4.1f}% | "
                f"len {plen:5.1f}/{tlen:5.1f} | {rate:6.0f} lines/s | {self.seen:>8,} lines | {L.fmt_s(el)} el, ETA {L.fmt_s(eta)} | "
                f"mem {gpu_mem()}" + (f" | {self.skipped} overflow steps skipped (normal for a few early AMP steps)"
                                      if self.skipped else ""))
            self.last_log = time.time()
        return lv


def batches_from(store, idx_batches, targets, aug=False, rng=None, pin=False):
    """Generator of (x, lens, y, ylens) tensors for the given batches of store indices."""
    for b in idx_batches:
        imgs = [store.img(i) for i in b]
        if aug:
            imgs = [L.augment_line(im, rng) for im in imgs]
        x, lens = L.to_batch(imgs, store.H)
        ys = [targets[i] for i in b]
        y = np.fromiter((t for yy in ys for t in yy), np.int64)
        yl = np.array([len(yy) for yy in ys], np.int64)
        xt = torch.from_numpy(x)
        if pin:
            xt = xt.pin_memory()
        yield xt, torch.from_numpy(lens), torch.from_numpy(y), torch.from_numpy(yl)


def time_left():
    return DEADLINE - EVAL_RESERVE - time.time()


# ======================================================================= stages
def pretrain(model, P):
    seed_all(P.get("seed", 0))
    pool = L.LineStore(os.path.join(W, f"pool_{P['pool']}_h{H}"), P.get("limit"))
    tg = [vocab.encode(lab) for lab in pool.labels]
    ok = np.array([i for i in range(len(pool)) if L.line_ok(len(tg[i]), pool.widths[i])], np.int64)
    if len(ok) == 0:
        raise RuntimeError(f"pool {P['pool']} has no usable lines")
    batch = int(P.get("batch", 32)); steps = max(1, int(P["samples"]) // batch)
    fpc = pool.widths[ok] / 2 / np.maximum(1, np.array([len(tg[i]) for i in ok]))
    log(f"PRETRAIN on pool {P['pool']} (first {len(pool):,} lines; style {pool.meta.get('style')}, text {pool.meta.get('text')}) | "
        f"usable {len(ok):,} (dropped {len(pool) - len(ok)} too long for CTC) | {steps:,} steps x {batch} = {steps*batch:,} lines "
        f"= {steps*batch/max(1,len(ok)):.1f} passes | targets/line median {np.median([len(tg[i]) for i in ok]):.0f} | "
        f"CTC frames per target median {np.median(fpc):.1f}")
    st = Stage("pre", model, P.get("lr", 1e-3), steps, log_every=int(P.get("log_every", 100)))
    rng = np.random.default_rng(P.get("seed", 0) + 17)

    def gen():
        n = 0
        while n < steps:
            for b in L.batch_plan(pool.widths, ok, batch, rng):
                if n >= steps:
                    return
                yield b; n += 1

    marks = set(int(steps * k / max(1, P.get("evals", 4))) for k in range(1, int(P.get("evals", 4)) + 1))
    stream = L.Prefetch(batches_from(pool, gen(), tg, pin=dev.type == "cuda"))
    for bt in stream:
        st.update(bt)
        if st.step in marks or st.step == steps:
            m, refs, hyps = eval_patches(model, VAL_SEL)
            record("pre", st.step, st.seen, st.ema if st.ema is not None else float("nan"), st.sched.get_last_lr()[0], m)
            log(f"  zero-shot REAL val ({len(VAL_SEL)} patches) after {st.seen:,} synthetic lines: {fmt_m(m)}")
            j = VAL_SEL.index(SHOW)
            log(f"     GT  : {refs[j][:110]}\n     PRED: {hyps[j][:110]}")
        if time_left() < 0:
            log(f"  TIME CAP: pretraining stopped at step {st.step:,}/{steps:,} ({100*st.step/steps:.0f}% of plan)")
            RESULT["capped"] = True
            break
    RESULT["pre"] = dict(pool=P["pool"], limit=P.get("limit"), steps_done=st.step, steps_planned=steps, lines_seen=st.seen,
                         minutes=round((time.time() - st.t0) / 60, 2), final_loss_ema=st.ema, skipped=st.skipped,
                         lines_per_s=round(st.seen / max(1e-9, time.time() - st.t0), 1))
    log(f"PRETRAIN done: {st.step:,} steps, {st.seen:,} lines in {L.fmt_s(time.time()-st.t0)} "
        f"({RESULT['pre']['lines_per_s']:.0f} lines/s), loss ema {st.ema:.3f}")


def realign(model, line_labels):
    """X3 at GPU scale: re-split every multi-line TRAINING patch's label with the current model."""
    t0 = time.time()
    pats = [p for p in TRAIN_PATCHES if len(by_patch[p]) > 1]
    changed = total = skipped = 0; dl = []
    for s in range(0, len(pats), 48):
        chunk = pats[s:s + 48]
        _, lps, _ = predict(model, real, [by_patch[p] for p in chunk], want_lp=True)
        for p in chunk:
            ls = by_patch[p]
            pieces = L.realign_patch([lps[i] for i in ls], TEXT[p], vocab)
            if pieces is None:
                skipped += 1; continue
            for i, piece in zip(ls, pieces):
                total += 1
                if piece != line_labels[i]:
                    changed += 1; dl.append(abs(len(piece) - len(line_labels[i])))
                line_labels[i] = piece
    msg = dict(patches=len(pats), skipped=skipped, lines=total, changed=changed,
               changed_pct=round(100 * changed / max(1, total), 1), median_abs_len_change=float(np.median(dl)) if dl else 0.0,
               seconds=round(time.time() - t0, 1))
    log(f"  RE-ALIGNED line labels (CTC forced alignment): {changed:,}/{total:,} changed ({msg['changed_pct']}%), "
        f"median |length change| {msg['median_abs_len_change']:.0f} chars, {skipped} patches skipped | {msg['seconds']}s")
    return msg


def finetune(model, Fs):
    seed_all(Fs.get("seed", 0))
    epochs = int(Fs["epochs"]); batch = int(Fs.get("batch", 32))
    line_labels = list(real.labels)                        # mutable: re-alignment rewrites them
    rng = np.random.default_rng(Fs.get("seed", 0) + 29)
    mix = float(Fs.get("mix", 0.0))
    mix_store = mix_tg = mix_ok = None
    if mix > 0:
        mix_store = L.LineStore(os.path.join(W, f"pool_{Fs.get('mix_pool', 'A')}_h{H}"), Fs.get("mix_limit"))
        mix_tg = [vocab.encode(l) for l in mix_store.labels]
        mix_ok = np.array([i for i in range(len(mix_store)) if L.line_ok(len(mix_tg[i]), mix_store.widths[i])], np.int64)

    def targets():
        tg = {i: vocab.encode(line_labels[i]) for i in TRAIN_LINES}
        ok = np.array([i for i in TRAIN_LINES if L.line_ok(len(tg[i]), real.widths[i])], np.int64)
        return tg, ok

    tg, ok = targets()
    per_epoch = int(math.ceil(len(ok) / batch)) * (2 if mix > 0 else 1)
    realign_at = set(int(e) for e in Fs.get("realign_at", []))
    if realign_at and vocab.no_space:
        log("  re-alignment needs word spaces to snap to; not applicable to no-space targets - skipped")
        realign_at = set()
    log(f"FINE-TUNE on real lines: {len(ok):,} usable training lines (dropped {len(TRAIN_LINES) - len(ok)}) | {epochs} epochs x "
        f"{per_epoch} steps (batch {batch}{', +1 synthetic batch per real batch' if mix > 0 else ''}) | "
        f"augmentation {'ON' if Fs.get('aug') else 'off'} | re-align after epochs {sorted(realign_at) or 'never'} | "
        f"select by val {'no-space ' if vocab.no_space else ''}CER")
    st = Stage("ft", model, Fs.get("lr", 1e-3), epochs * per_epoch, warmup=100, log_every=int(Fs.get("log_every", 60)))
    best = dict(primary=1e9, epoch=0, state=None)
    ra_log = []
    for ep in range(1, epochs + 1):
        t_ep = time.time()
        plan = L.batch_plan(real.widths, ok, batch, rng)
        if mix > 0:
            syn = L.batch_plan(mix_store.widths, mix_ok[rng.permutation(len(mix_ok))[:len(ok)]], batch, rng)

            def interleave():
                for k, b in enumerate(plan):
                    yield ("r", b)
                    if k < len(syn):
                        yield ("s", syn[k])
            src = interleave()

            def gen():
                for kind, b in src:
                    if kind == "r":
                        yield next(batches_from(real, [b], tg, aug=bool(Fs.get("aug")), rng=rng, pin=dev.type == "cuda"))
                    else:
                        yield next(batches_from(mix_store, [b], mix_tg, pin=dev.type == "cuda"))
            stream = L.Prefetch(gen())
        else:
            stream = L.Prefetch(batches_from(real, plan, tg, aug=bool(Fs.get("aug")), rng=rng, pin=dev.type == "cuda"))
        tot = n = 0
        for bt in stream:
            lv = st.update(bt)
            if lv is not None:
                tot += lv; n += 1
            if time_left() < 0:
                break
        m, refs, hyps = eval_patches(model, VAL_SEL)
        improved = m["primary"] < best["primary"]
        if improved:
            best.update(primary=m["primary"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        record("ft", ep, st.seen, tot / max(1, n), st.sched.get_last_lr()[0], m, dict(best=round(best["primary"], 3)))
        log(f"ep {ep:3d}/{epochs} | train loss {tot/max(1,n):6.3f} | val {fmt_m(m)} | best {best['primary']:5.1f}@{best['epoch']} | "
            f"{time.time()-t_ep:5.1f}s/epoch | {L.fmt_s(time.time()-T_JOB)} job time" + ("  *BEST*" if improved else ""))
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            j = VAL_SEL.index(SHOW)
            log(f"     GT  : {refs[j][:110]}\n     PRED: {hyps[j][:110]}")
            an = L.edit_anatomy(refs, hyps)
            log(f"     errors: deletions {an['deletions_pct']}% insertions {an['insertions_pct']}% substitutions {an['substitutions_pct']}% | "
                f"recall of key marks {an['focus_char_recall']}")
        if ep in realign_at and time_left() > 60:
            ra_log.append(dict(epoch=ep, **realign(model, line_labels)))
            tg, ok = targets()
        if time_left() < 0:
            log(f"  TIME CAP: fine-tuning stopped after epoch {ep}/{epochs}")
            RESULT["capped"] = True
            break
    RESULT["ft"] = dict(epochs_done=ep, epochs_planned=epochs, best_epoch=best["epoch"], best_val_sel_primary=best["primary"],
                        minutes=round((time.time() - st.t0) / 60, 2), lines_seen=st.seen, skipped=st.skipped,
                        lines_per_s=round(st.seen / max(1e-9, time.time() - st.t0), 1), realign=ra_log,
                        train_lines=int(len(ok)), dropped=int(len(TRAIN_LINES) - len(ok)))
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    log(f"FINE-TUNE done: best val(sel) {'no-space ' if vocab.no_space else ''}CER {best['primary']:.2f} at epoch {best['epoch']} | "
        f"{L.fmt_s(time.time()-st.t0)}")


def stroke_stats(store, idxs):
    """Ink share and median horizontal ink run (stroke thickness proxy, px at this line height)."""
    inks, runs = [], []
    for i in idxs:
        b = store.img(i) < 128
        inks.append(b.mean())
        d = np.diff(np.pad(b.astype(np.int8), ((0, 0), (1, 1))), axis=1)
        st, en = np.where(d == 1), np.where(d == -1)
        if len(st[1]):
            runs.append(float(np.median(en[1] - st[1])))
    return float(np.mean(inks)) if inks else float("nan"), float(np.median(runs)) if runs else float("nan")


def n1n2(patches, refs, hyps):
    """N1: do deletions grow where there are few CTC frames per character?  N2: do errors cluster by writing
    size (line pitch), number of lines, ink density or stroke thickness?"""
    from rapidfuzz.distance import Levenshtein
    rows = []
    for p, r, h in zip(patches, refs, hyps):
        r2, h2 = r.replace("<FRAC>", "\x01"), h.replace("<FRAC>", "\x01")
        ops = Levenshtein.editops(r2, h2)
        n = max(1, len(r2))
        frames = sum(math.ceil(real.widths[i] / 2) for i in by_patch.get(p, []))
        ink, stroke = stroke_stats(real, by_patch.get(p, []))
        rows.append(dict(patch=p, cer=100 * len(ops) / n, del_rate=100 * sum(o.tag == "delete" for o in ops) / n,
                         ins_rate=100 * sum(o.tag == "insert" for o in ops) / n,
                         frames_per_char=frames / max(1, len(r2.replace(" ", ""))), pitch=PINFO[p]["pitch"],
                         n_lines=PINFO[p]["n_lines"], label_len=len(r2), ink=ink, stroke=stroke))
    import pandas as pd
    d = pd.DataFrame(rows); d.to_csv(os.path.join(RUN, "n1n2_per_patch.csv"), index=False)
    out = dict(n=len(d))
    for f in ["frames_per_char", "pitch", "n_lines", "label_len", "ink", "stroke"]:
        q = pd.qcut(d[f].rank(method="first"), 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
        out[f] = dict(spearman_cer=round(L.spearman(d[f], d.cer), 3), spearman_del=round(L.spearman(d[f], d.del_rate), 3),
                      cer_by_quartile={str(k): round(v, 1) for k, v in d.groupby(q, observed=True).cer.mean().items()},
                      del_by_quartile={str(k): round(v, 1) for k, v in d.groupby(q, observed=True).del_rate.mean().items()})
    return out


def inside_figure(model, patches, path, n=4):
    """What the network does inside, frame by frame: under each line image, the probability of blank (grey) and of
    its best character (red). Each red spike is where it writes a character; a line with no spikes reads nothing."""
    try:
        import warnings
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        groups = [by_patch[p] for p in patches if by_patch.get(p)][:n]
        if not groups:
            return
        idxs = [g[0] for g in groups]                      # first line of each patch, read with its own patch
        _, lps, _ = predict(model, real, groups, want_lp=True)
        fig, axes = plt.subplots(2 * len(idxs), 1, figsize=(16, 2.2 * len(idxs)), squeeze=False)
        conf = []
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for k, i in enumerate(idxs):
                img, pr = real.img(i), np.exp(lps[i])
                ax = axes[2 * k, 0]
                ax.imshow(img, cmap="gray", aspect="auto", extent=(0, img.shape[1], img.shape[0], 0)); ax.axis("off")
                emit = pr[:, 1:].max(1)[pr.argmax(1) != 0]
                conf += list(emit)
                ax.set_title(f"{real.info[i]['patch']} line {real.info[i]['idx'] + 1}: {len(emit)} characters emitted, "
                             f"mean confidence {emit.mean() if len(emit) else 0:.2f}, blank on {100 * (pr.argmax(1) == 0).mean():.0f}% of frames",
                             fontsize=8, loc="left")
                ax2 = axes[2 * k + 1, 0]; x = np.arange(len(pr)) * L.STRIDE
                ax2.plot(x, pr[:, 0], color="0.6", lw=0.8, label="blank")
                ax2.plot(x, pr[:, 1:].max(1), color="tab:red", lw=0.9, label="best character")
                ax2.set_xlim(0, img.shape[1]); ax2.set_ylim(0, 1.02); ax2.legend(fontsize=7, loc="upper right")
        fig.tight_layout(); fig.savefig(path, dpi=80); plt.close(fig)
        log(f"  inside the network: {len(conf)} characters emitted on {len(idxs)} val lines, mean confidence "
            f"{np.mean(conf) if conf else 0:.2f} -> {os.path.basename(path)}")
    except Exception as e:
        log(f"  (inside-the-network figure skipped: {e})")


def final_eval(model):
    t0 = time.time()
    res = {}
    allv = VAL_SEL + VAL_REP
    m1, r1, h1 = eval_patches(model, VAL_SEL, save="val_sel_predictions.csv")
    m2, r2, h2 = eval_patches(model, VAL_REP, save="val_rep_predictions.csv")
    m3, _, _ = eval_patches(model, TR150, save="train150_predictions.csv")
    res.update(val_sel=m1, val_rep=m2, train150=m3)
    res["anatomy"] = L.edit_anatomy(r1 + r2, h1 + h2)
    res["n1n2"] = n1n2(allv, r1 + r2, h1 + h2)
    log("=" * 110)
    log(f"FINAL [{S['tag']}] best checkpoint, greedy decoding")
    log(f"  VAL-SEL ({len(VAL_SEL)} patches, used to pick the epoch): {fmt_m(m1)}")
    log(f"  VAL-REP ({len(VAL_REP)} patches, never used for any choice): {fmt_m(m2)}")
    log(f"  TRAIN   ({len(TR150)} patches; reading vs memorising): {fmt_m(m3)}")
    a = res["anatomy"]
    log(f"  errors on all val: {a['errors']:,} = deletions {a['deletions_pct']}% / insertions {a['insertions_pct']}% / "
        f"substitutions {a['substitutions_pct']}%")
    log(f"  most missed: {a['most_missed']}")
    log(f"  top confusions: {a['top_confusions'][:10]}")
    log(f"  recall of key marks: {a['focus_char_recall']}")
    for f, v in res["n1n2"].items():
        if isinstance(v, dict):
            log(f"  N1/N2 {f:16s}: Spearman with CER {v['spearman_cer']:+.2f}, with deletion rate {v['spearman_del']:+.2f} | "
                f"CER by quartile {v['cer_by_quartile']}")
    for k in range(min(3, len(r2))):
        log(f"   GT  : {r2[k][:110]}\n   PRED: {h2[k][:110]}")
    inside_figure(model, VAL_REP, os.path.join(RUN, "inside_alignment.png"))
    log(f"  (final evaluation {time.time()-t0:.0f}s)")
    log("=" * 110)
    return res


def bench():
    """Throughput of each model shape on this GPU, so the notebook can size every job to the time budget."""
    out = {}
    for cfg in S["bench"]:
        st_store = L.LineStore(os.path.join(W, f"real_h{cfg['H']}"))
        bp = {}
        for i, inf in enumerate(st_store.info):
            bp.setdefault(inf["patch"], []).append(i)
        model = L.build_crnn(len(vocab), cfg["size"]).to(dev)
        seed_all(0)
        tg = {i: vocab.encode(st_store.labels[i]) for i in range(len(st_store)) if st_store.info[i]["split"] == "train"}
        ok = np.array([i for i in tg if L.line_ok(len(tg[i]), st_store.widths[i])], np.int64)
        n_warm, n_time = int(cfg.get("warm", 5)), int(cfg.get("steps", 25))
        plan = L.batch_plan(st_store.widths, ok, 32, np.random.default_rng(0))
        plan = (plan * ((n_warm + n_time) // max(1, len(plan)) + 1))[:n_warm + n_time]     # cycle if the set is tiny
        stg = Stage("bench", model, 1e-3, 1000, log_every=10 ** 9)
        it = iter(L.Prefetch(batches_from(st_store, plan, tg, pin=dev.type == "cuda")))
        for _ in range(n_warm):
            stg.update(next(it))
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.time(); n0 = stg.seen
        for _ in range(n_time):
            stg.update(next(it))
        if dev.type == "cuda": torch.cuda.synchronize()
        rate = (stg.seen - n0) / max(1e-9, time.time() - t0)
        model.eval()
        t1 = time.time()
        idxs = [i for p in VAL_SEL for i in bp.get(p, [])]
        with torch.no_grad():
            for s in range(0, len(idxs), 64):
                b = idxs[s:s + 64]
                x, _ = L.to_batch([st_store.img(i) for i in b], st_store.H)
                with autocast():
                    model(torch.from_numpy(x).to(dev))
        if dev.type == "cuda": torch.cuda.synchronize()
        ev = time.time() - t1
        key = f"{cfg['size']}_h{cfg['H']}"
        out[key] = dict(lines_per_s=round(rate, 1), eval_s=round(ev + 1.0, 2),
                        params_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 3), mem=gpu_mem())
        log(f"BENCH {key}: {rate:,.0f} training lines/s (batch 32, AMP {'on' if AMP else 'off'}) | val forward pass "
            f"{ev:.1f}s | {out[key]['params_M']} M params | peak mem {gpu_mem()}")
        del model
        if dev.type == "cuda": torch.cuda.empty_cache()
    json.dump(out, open(os.path.join(RUN, "bench.json"), "w"), indent=2)
    RESULT["bench"] = out


def main():
    log(f"JOB {S['tag']} | hypotheses {S.get('hyp')} | {S.get('desc','')}")
    log(f"  device {dev} ({torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu'}) | torch {torch.__version__} | "
        f"AMP {'on' if AMP else 'off'} | deadline in {L.fmt_s(DEADLINE - time.time())}")
    log(f"  model {S.get('size')} | line height {H} px | targets {vocab.describe()} | init {os.path.basename(S.get('init') or '') or 'random'}")
    if S.get("bench"):
        bench(); RESULT["status"] = "ok"; return
    seed_all(int((S.get("pretrain") or {}).get("seed", 0)))
    model = L.build_crnn(len(vocab), S["size"]).to(dev)
    RESULT["params_M"] = round(sum(p.numel() for p in model.parameters()) / 1e6, 3)
    log(f"  {RESULT['params_M']} M parameters | {len(TRAIN_LINES):,} real training lines in {len(TRAIN_PATCHES):,} patches | "
        f"val-sel {len(VAL_SEL)} / val-rep {len(VAL_REP)} patches")
    if S.get("init"):
        sd = torch.load(S["init"], map_location="cpu")
        model.load_state_dict(sd["model"] if "model" in sd else sd, strict=True)
        log(f"  loaded {S['init']} (strict)")
    if S.get("pretrain"):
        pretrain(model, S["pretrain"])
        if S.get("save_pre"):
            torch.save(dict(model=model.state_dict(), size=S["size"], H=H, vocab=vocab.describe()), os.path.join(RUN, "pre.pt.tmp"))
            os.replace(os.path.join(RUN, "pre.pt.tmp"), os.path.join(RUN, "pre.pt"))
            log(f"  saved {os.path.join(RUN, 'pre.pt')}")
    if S.get("finetune"):
        finetune(model, S["finetune"])
    torch.save(dict(model=model.state_dict(), size=S["size"], H=H, vocab=vocab.describe(), tag=S["tag"]),
               os.path.join(RUN, "best.pt"))
    RESULT.update(final_eval(model))
    RESULT["status"] = "ok"


if __name__ == "__main__":
    try:
        main()
    except Exception:
        RESULT["status"] = "failed"; RESULT["error"] = traceback.format_exc()
        log("JOB FAILED:\n" + RESULT["error"])
    RESULT["job_minutes"] = round((time.time() - T_JOB) / 60, 2)
    if dev.type == "cuda":
        RESULT["peak_gpu_mem"] = gpu_mem()
    with open(os.path.join(RUN, "result.json.tmp"), "w", encoding="utf-8") as f:
        json.dump(RESULT, f, indent=2, ensure_ascii=False, default=str)
    os.replace(os.path.join(RUN, "result.json.tmp"), os.path.join(RUN, "result.json"))
    log(f"JOB {S['tag']} {RESULT['status'].upper()} in {RESULT['job_minutes']} min")
    sys.exit(0 if RESULT["status"] == "ok" else 1)

In [ ]:
%%writefile /kaggle/working/lr05_code/lr05_run.py
"""lr05_run.py - the experiment matrix, the time planner and the two-GPU scheduler of notebook 05.

Every job changes ONE thing relative to the anchor (FT_A0_s0 = 60k plain synthetic lines -> 40 real epochs),
so each comparison answers one question. Jobs run as separate processes, one per GPU; a free GPU always
takes the highest-priority job whose inputs (a synthetic pool, a pretrained checkpoint) are ready, and a
job is only started if its time estimate fits before the deadline.
"""
import json, math, os, subprocess, sys, time
import lr05_lib as L

# ---------------------------------------------------------------- the questions (also used by the report)
HYPOTHESES = [
    # id, question, variant, control, metric
    ("H24-GPU", "Can GPU-scale training on real lines alone (no synthetic pretraining) learn to read?", "NOPRE", "FT_A0_s0", "CER"),
    ("N10a", "Fewer distinct synthetic lines (12k instead of 60k, same number of steps) - worse?", "A12", "FT_A0_s0", "CER"),
    ("N10b", "More distinct synthetic lines (120k instead of 60k, same steps) - better?", "A120", "FT_A0_s0", "CER"),
    ("N10c", "Three times longer pretraining on 120k lines - better than the normal length?", "PRE_LONG", "A120", "CER"),
    ("N4", "Realistic rendering (no word gaps + continuous headline) instead of plain print - better?", "B60", "FT_A0_s0", "CER"),
    ("N12b", "Synthetic text sampled from a character LM (new word forms) instead of training-label text - better?", "C60", "FT_A0_s0", "CER"),
    ("N11", "A 2x wider model (4x the parameters) - better now that learning works?", "BIG", "FT_A0_s0", "CER"),
    ("N14", "Akshara (syllable) targets instead of Unicode characters - better?", "AKS", "FT_A0_s0", "CER"),
    ("N5", "No spaces in the targets (Modi has no word gaps) - better at the letters themselves?", "NS", "FT_A0_s0", "CER_nospace"),
    ("N7", "64-px lines instead of 40-px (small marks: virama, anusvara) - better?", "H64", "FT_A0_s0", "CER"),
    ("N9", "Keep synthetic lines in the mix during fine-tuning - better than real lines only?", "MIX", "FT_A0_s0", "CER"),
    ("N6", "Augment real lines during fine-tuning - better?", "AUG", "FT_A0_s0", "CER"),
    ("H9/N8a", "One CTC re-alignment of the line labels during fine-tuning - better?", "EM1", "FT_A0_s0", "CER"),
    ("N8b", "A second re-alignment round - better than one?", "EM2", "EM1", "CER"),
    ("H11-line", "Three times more fine-tuning epochs - still improving?", "FT_LONG", "FT_A0_s0", "CER"),
    ("COMBO", "All winning changes together (normal length) - better than the anchor?", "COMBO", "FT_A0_s0", "CER"),
    ("FINAL", "The final recipe trained as long as the budget allows - how much better than at normal length?", "FINAL_s0", "COMBO", "CER"),
]


def experiment_matrix(runs, base, sizes):
    """sizes: pool sizes and the subsets the experiments use (full run: A 200k with milestones 12k/60k/120k, B and C
    120k with a 60k milestone). Comparisons always use equal-size subsets; only FINAL uses a whole pool."""
    pre = lambda pool, limit, samples=None, H=40: dict(pool=pool, limit=limit, samples=int(samples or base["pre_samples"]),
                                                       lr=1e-3, batch=base["batch"], evals=4, seed=0)
    ft = lambda **kw: dict(dict(epochs=base["ft_epochs"], lr=1e-3, batch=base["batch"], aug=False, mix=0.0,
                                realign_frac=[], seed=0), **kw)
    PTA = os.path.join(runs, "PT_A60", "pre.pt")
    a_small, a_mid, a_big, b_n, c_n = sizes["A_small"], sizes["A_mid"], sizes["A_big"], sizes["B_mid"], sizes["C_mid"]
    J = []

    def add(tag, pri, hyp, desc, size="small", H=40, vocab="char", no_space=False, init=None, pretrain=None,
            finetune=None, save_pre=False, after=(), pools=()):
        J.append(dict(tag=tag, priority=pri, hyp=list(hyp), desc=desc, size=size, H=H, vocab=vocab, no_space=no_space,
                      init=init, pretrain=pretrain, finetune=finetune, save_pre=save_pre, after=list(after),
                      pools=[list(p) for p in pools]))

    # priority 1: the anchor, the decisive control, and the cheapest high-value questions
    add("PT_A60", 1, ["anchor"], f"shared pretraining: {a_mid:,} plain synthetic lines (X4 style)",
        pretrain=pre("A", a_mid), save_pre=True, pools=[("A", 40, a_mid)])
    add("FT_A0_s0", 1, ["anchor"], "ANCHOR: PT_A60 -> real lines, proportional line labels, seed 0",
        init=PTA, finetune=ft(seed=0), after=["PT_A60"])
    add("NOPRE", 1, ["H24-GPU"], "CONTROL: real lines only, no synthetic pretraining, 2x the epochs",
        finetune=ft(epochs=base["nopre_epochs"]))
    add("A12", 1, ["N10a"], f"pretrain on only {a_small:,} distinct synthetic lines (same steps as the anchor)",
        pretrain=pre("A", a_small), finetune=ft(), pools=[("A", 40, a_small)])
    add("B60", 1, ["N4"], f"pretrain on {b_n:,} REALISTIC synthetic lines (no word gaps + continuous headline)",
        pretrain=pre("B", b_n), finetune=ft(), pools=[("B", 40, b_n)])
    add("EM1", 1, ["H9/N8a"], "anchor + one CTC re-alignment of the line labels after 1/3 of fine-tuning",
        init=PTA, finetune=ft(realign_frac=[1 / 3]), after=["PT_A60"])
    add("FT_A0_s1", 1, ["seeds"], "anchor, fine-tune seed 1 (measures run-to-run noise)", init=PTA, finetune=ft(seed=1), after=["PT_A60"])
    # priority 2
    add("A120", 2, ["N10b"], f"pretrain on {a_big:,} distinct synthetic lines (same steps)", pretrain=pre("A", a_big),
        finetune=ft(), pools=[("A", 40, a_big)])
    add("C60", 2, ["N12b"], f"pretrain on {c_n:,} lines whose TEXT is sampled from a char 4-gram of the training labels",
        pretrain=pre("C", c_n), finetune=ft(), pools=[("C", 40, c_n)])
    add("BIG", 2, ["N11"], "2x wider CRNN (conv 64-128-256-256-256, BiLSTM 2x256)", size="big",
        pretrain=pre("A", a_mid), finetune=ft(), pools=[("A", 40, a_mid)])
    add("AKS", 2, ["N14"], "akshara (syllable) targets, pretrain + fine-tune", vocab="akshara",
        pretrain=pre("A", a_mid), finetune=ft(), pools=[("A", 40, a_mid)])
    add("NS", 2, ["N5"], "no spaces in any target, pretrain + fine-tune (spaces put back by the LM afterwards)",
        no_space=True, pretrain=pre("A", a_mid), finetune=ft(), pools=[("A", 40, a_mid)])
    add("MIX", 2, ["N9"], "anchor fine-tune with one synthetic batch after every real batch",
        init=PTA, finetune=ft(mix=0.5, mix_pool="A", mix_limit=a_mid), after=["PT_A60"], pools=[("A", 40, a_mid)])
    add("AUG", 2, ["N6"], "anchor fine-tune with augmented real lines", init=PTA, finetune=ft(aug=True), after=["PT_A60"])
    add("EM2", 2, ["N8b"], "anchor + re-alignment after 1/3 and 2/3 of fine-tuning", init=PTA,
        finetune=ft(realign_frac=[1 / 3, 2 / 3]), after=["PT_A60"])
    add("FT_A0_s2", 2, ["seeds"], "anchor, fine-tune seed 2", init=PTA, finetune=ft(seed=2), after=["PT_A60"])
    # priority 3: run if time allows
    add("H64", 3, ["N7"], "64-px line images (pretrain + fine-tune)", H=64,
        pretrain=pre("A", min(a_mid, sizes["A_h64"])), finetune=ft(), pools=[("A", 64, min(a_mid, sizes["A_h64"]))])
    add("FT_LONG", 3, ["H11-line"], "anchor fine-tune for 3x the epochs", init=PTA,
        finetune=ft(epochs=base["long_ft_epochs"]), after=["PT_A60"])
    add("PRE_LONG", 3, ["N10c"], f"3x longer pretraining on {a_big:,} lines", pretrain=pre("A", a_big, base["long_pre_samples"]),
        finetune=ft(), pools=[("A", 40, a_big)])
    return J


def estimate_s(j, bench, n_train_lines):
    b = bench.get(f"{j['size']}_h{j['H']}") or bench.get("small_h40")
    thr, ev = max(1.0, b["lines_per_s"]), b["eval_s"]
    t = 40.0                                                             # start-up + loading
    if j.get("pretrain"):
        t += j["pretrain"]["samples"] / thr + j["pretrain"]["evals"] * ev
    if j.get("finetune"):
        Fz = j["finetune"]
        t += Fz["epochs"] * n_train_lines * (2 if Fz.get("mix", 0) > 0 else 1) / thr + Fz["epochs"] * ev
        t += len(Fz.get("realign_frac", [])) * (6 * ev + 15)
    t += 4 * ev + 15                                                     # final evaluation
    return t * (1.06 if j["vocab"] == "akshara" else 1.0)


def apply_scale(jobs, scale, base):
    for j in jobs:
        if j.get("pretrain"):
            j["pretrain"]["samples"] = max(j["pretrain"]["batch"] * 20, int(j["pretrain"]["samples"] * scale))
        if j.get("finetune"):
            j["finetune"]["epochs"] = max(min(4, j["finetune"]["epochs"]), int(round(j["finetune"]["epochs"] * scale)))
    return jobs


def finalize_spec(j, work, runs, deadline, est):
    s = dict(j)
    s.update(work=work, run_dir=os.path.join(runs, j["tag"]), deadline=deadline, time_cap_s=est * 1.8 + 180)
    if s.get("finetune"):
        Fz = dict(s["finetune"]); E = Fz["epochs"]
        Fz["realign_at"] = sorted(set(max(1, min(E - 1, int(round(f * E)))) for f in Fz.pop("realign_frac", [])))
        s["finetune"] = Fz
    return s


class Scheduler:
    def __init__(self, jobs, gpus, script, work, runs, deadline, bench, n_train_lines, log, poll=10.0,
                 renderer=None, reserve=None, cpu=False, threads=2, tails=(), known=None, ratios=None):
        self.jobs = {j["tag"]: j for j in jobs}
        self.order = [j["tag"] for j in sorted(jobs, key=lambda j: j["priority"])]
        self.state = {t: "pending" for t in self.order}
        self.gpus, self.script, self.work, self.runs = list(gpus), script, work, runs
        self.deadline, self.bench, self.ntl, self.log, self.poll = deadline, bench, n_train_lines, log, poll
        self.renderer, self.reserve = renderer, (reserve or {})
        self.cpu, self.threads = cpu, threads
        self.running = {}                         # gpu -> dict(tag, proc, fh, pos, t0, est)
        self.t_dash = 0.0
        self.notes = {}
        self.tails = [dict(path=p, tag=t, pos=0) for p, t in tails]     # other logs to stream (the renderer)
        self.known = dict(known or {})            # states of jobs finished by an earlier scheduler pass
        self.ratios = ratios if ratios is not None else []    # actual / estimated time of finished jobs (shared)

    def factor(self):
        """Correction learned from finished jobs: median of actual/estimated time, clamped to 0.5-2x."""
        r = sorted(self.ratios)
        return min(2.0, max(0.5, r[len(r) // 2])) if len(r) >= 2 else 1.0

    def est(self, tag):
        return estimate_s(self.jobs[tag], self.bench, self.ntl) * self.factor()

    def _deps(self, tag):
        j = self.jobs[tag]
        for t in j["after"]:
            st = self.state.get(t) or self.known.get(t, "missing")
            if st in ("failed", "skipped", "missing"):
                return "dead", f"needs {t}, which {st}"
            if st != "ok":
                return "wait", f"waiting for {t}"
        for name, H, need in j["pools"]:
            base = os.path.join(self.work, f"pool_{name}_h{H}")
            if not L.store_ready(base, need):
                if self.renderer is not None and self.renderer.poll() is not None:
                    return "dead", f"pool {name} (h{H}) never reached {need:,} lines"
                return "wait", f"waiting for pool {name} >= {need:,} lines"
        return "ready", ""

    def _launch(self, tag, gpu):
        est = self.est(tag)
        spec = finalize_spec(self.jobs[tag], self.work, self.runs, self.deadline, est)
        if self.cpu:
            spec.update(cpu=True, threads=self.threads)
        os.makedirs(spec["run_dir"], exist_ok=True)
        sp = os.path.join(spec["run_dir"], "spec.json")
        json.dump(spec, open(sp, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        out = os.path.join(spec["run_dir"], "stdout.txt")
        fh = open(out, "w", encoding="utf-8")
        env = dict(os.environ, PYTHONIOENCODING="utf-8")
        p = subprocess.Popen([sys.executable, "-u", self.script, "--spec", sp] + ([] if self.cpu else ["--gpu", str(gpu)]),
                             stdout=fh, stderr=subprocess.STDOUT, env=env)
        self.running[gpu] = dict(tag=tag, proc=p, fh=fh, path=out, pos=0, t0=time.time(), est=est,
                                 est_raw=estimate_s(self.jobs[tag], self.bench, self.ntl))
        self.state[tag] = "running"
        self.log(f">>> START {tag} on {'cpu-slot' if self.cpu else 'GPU'} {gpu} | P{self.jobs[tag]['priority']} | "
                 f"{self.jobs[tag]['desc']} | estimated {L.fmt_s(est)} | {L.fmt_s(self.deadline - time.time())} left")

    def _drain(self, r):
        try:
            with open(r["path"], "r", encoding="utf-8", errors="replace") as f:
                f.seek(r["pos"]); chunk = f.read(); r["pos"] = f.tell()
        except FileNotFoundError:
            return
        for line in chunk.splitlines():
            self.log(f"[{r['tag']}] {line}")

    def _reap(self):
        for gpu, r in list(self.running.items()):
            if r["proc"].poll() is None:
                if time.time() > self.deadline + 600:              # backstop; jobs obey the deadline themselves
                    r["proc"].kill()
                continue
            self._drain(r); r["fh"].close()
            res = os.path.join(self.runs, r["tag"], "result.json")
            ok = os.path.exists(res) and json.load(open(res, encoding="utf-8")).get("status") == "ok"
            self.state[r["tag"]] = "ok" if ok else "failed"
            took = time.time() - r["t0"]
            if ok:
                self.ratios.append(took / max(1.0, r["est_raw"]))
            self.log(f"<<< {'DONE' if ok else 'FAILED'} {r['tag']} in {L.fmt_s(took)} (estimated {L.fmt_s(r['est'])}, "
                     f"ratio {took / max(1, r['est']):.2f}) | time estimates now x{self.factor():.2f} (learned from {len(self.ratios)} jobs)")
            del self.running[gpu]

    def dashboard(self, force=False):
        if not force and time.time() - self.t_dash < 300:
            return
        self.t_dash = time.time()
        cnt = {s: sum(1 for v in self.state.values() if v == s) for s in ("ok", "failed", "skipped", "running", "pending")}
        self.log("-" * 110)
        self.log(f"DASHBOARD {time.strftime('%H:%M:%S')} | {L.fmt_s(self.deadline - time.time())} left before the deadline | "
                 f"done {cnt['ok']} | failed {cnt['failed']} | skipped {cnt['skipped']} | running {cnt['running']} | waiting {cnt['pending']}")
        for gpu, r in self.running.items():
            self.log(f"   slot {gpu}: {r['tag']} running {L.fmt_s(time.time() - r['t0'])} of ~{L.fmt_s(r['est'])}")
        waiting = [f"{t} ({self._deps(t)[1] or 'ready'})" for t in self.order if self.state[t] == "pending"]
        if waiting:
            self.log(f"   queue: {', '.join(waiting[:12])}")
        self.log(f"   {L.gpu_status()}")
        try:
            import psutil
            vm = psutil.virtual_memory(); du = psutil.disk_usage(self.work)
            self.log(f"   RAM {vm.used / 2**30:.1f}/{vm.total / 2**30:.1f} GB | disk free {du.free / 2**30:.0f} GB | CPU load {os.getloadavg()[0]:.1f}"
                     if hasattr(os, "getloadavg") else f"   RAM {vm.used / 2**30:.1f}/{vm.total / 2**30:.1f} GB")
        except Exception:
            pass
        self.log("-" * 110)

    def run(self):
        while True:
            self._reap()
            for gpu in self.gpus:
                if gpu in self.running:
                    continue
                for tag in self.order:
                    if self.state[tag] != "pending":
                        continue
                    st, why = self._deps(tag)
                    if st == "dead":
                        self.state[tag] = "skipped"; self.notes[tag] = why
                        self.log(f"--- SKIP {tag}: {why}")
                        continue
                    left = self.deadline - self.reserve.get(self.jobs[tag]["priority"], 0) - time.time()
                    if self.est(tag) > left:
                        if st == "ready" or self.est(tag) > self.deadline - time.time():
                            self.state[tag] = "skipped"; self.notes[tag] = f"not enough time (needs ~{L.fmt_s(self.est(tag))}, {L.fmt_s(max(0, left))} left)"
                            self.log(f"--- SKIP {tag}: {self.notes[tag]}")
                        continue
                    if st == "ready":
                        self._launch(tag, gpu)
                        break
            for r in list(self.running.values()) + self.tails:
                self._drain(r)
            self.dashboard()
            if not self.running and not any(s == "pending" for s in self.state.values()):
                break
            if not self.running and all(self._deps(t)[0] == "wait" for t in self.order if self.state[t] == "pending"):
                if time.time() > self.deadline:
                    for t in self.order:
                        if self.state[t] == "pending":
                            self.state[t] = "skipped"; self.notes[t] = "deadline passed while waiting"
                    break
            time.sleep(self.poll)
        self.dashboard(force=True)
        return dict(self.state)


# ---------------------------------------------------------------- the adaptive combination
def combo_jobs(results, base, sizes, runs, margin, log):
    """Build COMBO from every factor whose variant beat its control on VAL-SEL by more than `margin` CER points
    (VAL-REP stays untouched for the report). Returns (jobs, explanation)."""
    def sel(tag):
        r = results.get(tag)
        return r["val_sel"]["primary"] if r and r.get("status") == "ok" else None
    anchor = sel("FT_A0_s0")
    if anchor is None:
        return [], ["anchor missing - no combination"]
    why, win = [], {}
    pools = {"A60": anchor, "A120": sel("A120"), "B60": sel("B60"), "C60": sel("C60")}
    best_pool = min((k for k in pools if pools[k] is not None), key=lambda k: pools[k])
    if best_pool != "A60" and pools[best_pool] < anchor - margin:
        win["pool"] = best_pool
    for factor, tag in [("big", "BIG"), ("akshara", "AKS"), ("h64", "H64"), ("mix", "MIX"), ("aug", "AUG")]:
        v = sel(tag)
        if v is not None and v < anchor - margin:
            win[factor] = True
    em = {k: sel(k) for k in ("EM1", "EM2")}
    em_best = min((k for k in em if em[k] is not None), key=lambda k: em[k], default=None)
    if em_best and em[em_best] < anchor - margin:
        win["realign"] = em_best
    for k, v in sorted(pools.items()):
        why.append(f"pool {k}: val-sel CER {v if v is None else round(v, 2)}")
    for tag in ("BIG", "AKS", "H64", "MIX", "AUG", "EM1", "EM2"):
        why.append(f"{tag}: val-sel {sel(tag) if sel(tag) is None else round(sel(tag), 2)} vs anchor {anchor:.2f}")
    why.append(f"winning changes (better than the anchor by > {margin:.2f} CER on val-sel): {win or 'none'}")
    # COMBO uses exactly the subset the winning experiment used; FINAL uses the WHOLE pool of the winning style
    pool_name, limit = {"A60": ("A", sizes["A_mid"]), "A120": ("A", sizes["A_big"]), "B60": ("B", sizes["B_mid"]),
                        "C60": ("C", sizes["C_mid"])}[win.get("pool", "A60")]
    H = 64 if win.get("h64") else 40
    if H == 64:
        pool_name, limit = "A", min(sizes["A_mid"], sizes["A_h64"])
    full = sizes["A_h64"] if H == 64 else sizes[pool_name]

    def recipe(lim, seed):
        pre = dict(pool=pool_name, limit=lim, samples=int(base["pre_samples"]), lr=1e-3, batch=base["batch"], evals=4, seed=seed)
        ft = dict(epochs=base["ft_epochs"], lr=1e-3, batch=base["batch"], aug=bool(win.get("aug")),
                  mix=0.5 if win.get("mix") else 0.0, mix_pool="A", mix_limit=sizes["A_mid"], seed=seed,
                  realign_frac={"EM1": [1 / 3], "EM2": [1 / 3, 2 / 3]}.get(win.get("realign"), []))
        common = dict(size="big" if win.get("big") else "small", H=H, vocab="akshara" if win.get("akshara") else "char",
                      no_space=False, init=None, save_pre=False, after=[], pools=[[pool_name, H, lim]])
        if ft["mix"] > 0:
            common["pools"].append(["A", H, sizes["A_mid"]])
        return pre, ft, common

    desc = "COMBINATION of " + (", ".join(f"{k}={v}" for k, v in win.items()) if win else "nothing (no change beat the anchor)")
    jobs = []
    if win:
        pre, ft, common = recipe(limit, 0)
        jobs.append(dict(tag="COMBO", priority=4, hyp=["COMBO"], desc=desc + " - normal length", pretrain=pre, finetune=ft, **common))
    for s in (0, 1):
        pre, ft, common = recipe(full, s)
        jobs.append(dict(tag=f"FINAL_s{s}", priority=4, hyp=["FINAL"],
                         desc=f"FINAL model, seed {s}: {'the winning recipe' if win else 'the anchor recipe'} on the whole "
                              f"{pool_name} pool ({full:,} lines), trained as long as the time allows", pretrain=pre, finetune=ft, **common))
    return jobs, why


def fill_to(j, target_s, bench, ntl, factor=1.0, max_ft_mult=2.5):
    """Lengthen a job (never shorten it) until its estimated duration is ~target_s: fine-tuning up to max_ft_mult x
    its epochs, all remaining time to pretraining (synthetic data is unlimited, real data is not)."""
    est = estimate_s(j, bench, ntl) * factor
    if est >= target_s:
        return j
    if j.get("finetune"):
        j["finetune"]["epochs"] = int(round(j["finetune"]["epochs"] * min(target_s / est, max_ft_mult)))
    if j.get("pretrain"):
        b = bench.get(f"{j['size']}_h{j['H']}") or bench.get("small_h40")
        rest = target_s - estimate_s(dict(j, pretrain=dict(j["pretrain"], samples=0)), bench, ntl) * factor
        j["pretrain"]["samples"] = max(j["pretrain"]["samples"], int(rest / factor * b["lines_per_s"]))
    return j

In [ ]:
%%writefile /kaggle/working/lr05_code/lr05_final.py
"""lr05_final.py - after the jobs: hypothesis verdicts, character-LM decoding (N3), checkpoint ensembles (N15),
space restoration for the no-space model (N5), the one-time TEST score, figures and REPORT.md.

Rules fixed before the run (so nothing is tuned on the numbers it reports):
  * VAL-SEL (first 150 validation patches) picks epochs, the combination, LM weights and the final pipeline.
  * VAL-REP (the remaining validation patches) is only ever reported.
  * TEST is scored once, for exactly two pipelines: the anchor (+LM) and the pipeline VAL-SEL picks (+LM).
"""
import glob, json, math, os, pickle, time
import numpy as np
import lr05_lib as L
from lr05_run import HYPOTHESES


# ======================================================================= results and verdicts
def load_results(runs):
    R = {}
    for p in sorted(glob.glob(os.path.join(runs, "*", "result.json"))):
        if os.path.basename(os.path.dirname(p)).startswith("_"):      # _bench is a measurement, not an experiment
            continue
        try:
            r = json.load(open(p, encoding="utf-8"))
            R[r["tag"]] = r
        except Exception:
            pass
    return R


def preds(runs, tag, which):
    import pandas as pd
    p = os.path.join(runs, tag, f"{which}_predictions.csv")
    return pd.read_csv(p, keep_default_na=False) if os.path.exists(p) else None


def content_gap(runs, tag, which="val_rep"):
    """Reads or babbles? CER against each patch's own transcription vs against the patch next to it in transcription
    LENGTH. A reader is far better on its own patch (CPU line reader: 45.9 vs 78.6, better on 149/150); a model that only
    learned how much text a page holds is not (notebook 03: 73.7 vs 74.1; SPAN v1: 75.0 vs 75.6, a coin flip)."""
    import editdistance
    d = preds(runs, tag, which)
    if d is None or len(d) < 4:
        return None
    refs = [r.replace("<FRAC>", "") for r in d.ref]; hyps = [h.replace("<FRAC>", "") for h in d.hyp]
    order = np.argsort([len(r) for r in refs], kind="stable")
    own, nb = [], []
    for k, i in enumerate(order):
        j = order[k + 1] if k + 1 < len(order) else order[k - 1]
        own.append(editdistance.eval(refs[i], hyps[i]) / max(1, len(refs[i])))
        nb.append(editdistance.eval(refs[j], hyps[i]) / max(1, len(refs[j])))
    own, nb = np.array(own), np.array(nb)
    return dict(own=round(100 * own.mean(), 1), neighbour=round(100 * nb.mean(), 1),
                gap=round(100 * (nb.mean() - own.mean()), 1), own_better_pct=round(100 * float((own < nb).mean()), 1))


def compare(runs, a, b, which="val_rep", nospace=False):
    da, db = preds(runs, a, which), preds(runs, b, which)
    if da is None or db is None or list(da.patch) != list(db.patch):
        return None
    if nospace:
        return L.paired_bootstrap(da.err_nospace, db.err_nospace, da.n_nospace)
    return L.paired_bootstrap(da.err, db.err, da.n)


def seed_noise(R, which="val_rep", key="CER"):
    v = [R[t][which][key] for t in ("FT_A0_s0", "FT_A0_s1", "FT_A0_s2") if t in R and R[t].get("status") == "ok"]
    return (float(np.std(v, ddof=1)) if len(v) >= 2 else None), v


def verdicts(R, runs):
    sd, seeds = seed_noise(R)
    sd_ns, _ = seed_noise(R, key="CER_nospace")
    rows = []
    ok = lambda t: t in R and R[t].get("status") == "ok"
    for hid, q, var, ctl, metric in HYPOTHESES:
        if hid == "FINAL" and not ok("COMBO") and ok("FINAL_s0"):
            # nothing beat the anchor, so FINAL is the anchor recipe trained long: compare it with the anchor
            ctl, q = "FT_A0_s0", "No change beat the anchor, so: the anchor recipe on the whole pool, trained long - how much better than the anchor?"
        if not (ok(var) and ok(ctl)):
            why = f"not run ({var}: {R.get(var, {}).get('status', 'not run')}, {ctl}: {R.get(ctl, {}).get('status', 'not run')})"
            rows.append(dict(id=hid, question=q, variant=var, control=ctl, metric=metric, verdict="NOT TESTED", detail=why))
            continue
        c = compare(runs, var, ctl, nospace=(metric == "CER_nospace"))
        a, b = R[var]["val_rep"][metric], R[ctl]["val_rep"][metric]
        s = sd_ns if metric == "CER_nospace" else sd
        if c is None:
            rows.append(dict(id=hid, question=q, variant=var, control=ctl, metric=metric, verdict="NOT TESTED", detail="no predictions"))
            continue
        d, lo, hi = c
        thr = 2 * s if s is not None else 0.0
        if hi < 0 and d < -thr:
            v = "BETTER"
        elif lo > 0 and d > thr:
            v = "WORSE"
        else:
            v = "INCONCLUSIVE (within noise)"
        rows.append(dict(id=hid, question=q, variant=var, control=ctl, metric=metric, variant_value=round(a, 2),
                         control_value=round(b, 2), delta=round(d, 2), ci95=(round(lo, 2), round(hi, 2)),
                         seed_sd=None if s is None else round(s, 2), verdict=v,
                         detail=f"{metric} {a:.2f} vs {b:.2f}: {d:+.2f} (paired 95% CI {lo:+.2f} to {hi:+.2f}; "
                                f"seed noise sd {'n/a' if s is None else f'{s:.2f}'})"))
    return rows, dict(sd=sd, seeds=seeds)


# ======================================================================= log-probs for LM decoding
def _vocab_for(META, vd):
    return L.make_vocab(META["symbols"], META["train_texts"], vd["kind"], vd["no_space"])


def lm_inputs(runs, work, META, groups, patches, device, log, k=8):
    """groups: {key: [tags]} - one tag = a single model, several = an ensemble (probabilities averaged).
    Returns {key: dict(itos, sep, lines={i: (top_idx, top_lp)}, greedy={i: text}, by_patch={patch: [i]})}."""
    import torch
    out = {}
    stores = {}
    for key, tags in groups.items():
        ms, vocab, H = [], None, None
        for t in tags:
            ck = torch.load(os.path.join(runs, t, "best.pt"), map_location="cpu")
            v = _vocab_for(META, ck["vocab"])
            if vocab is not None and (v.itos != vocab.itos or ck["H"] != H):
                raise ValueError(f"ensemble {key}: {t} has a different alphabet or line height")
            vocab, H = v, ck["H"]
            m = L.build_crnn(len(v), ck["size"]); m.load_state_dict(ck["model"]); m.to(device).eval(); ms.append(m)
        if H not in stores:
            stores[H] = L.LineStore(os.path.join(work, f"real_h{H}"))
        st = stores[H]
        bp = {}
        for i, inf in enumerate(st.info):
            if inf["patch"] in patches:
                bp.setdefault(inf["patch"], []).append(i)
        for p in bp:
            bp[p].sort(key=lambda i: st.info[i]["idx"])
        lines, greedy = {}, {}
        with torch.no_grad():
            for b in bp.values():                      # one patch per batch, as in job.py and the CPU experiments
                x, lens = L.to_batch([st.img(i) for i in b], st.H, round_to=L.STRIDE)
                xt = torch.from_numpy(x).to(device)
                lps = [m(xt).float().log_softmax(-1) for m in ms]
                lp = lps[0] if len(lps) == 1 else torch.logsumexp(torch.stack(lps), 0) - math.log(len(lps))
                lp = lp.cpu().numpy()
                for kk, i in enumerate(b):
                    row = lp[kk, :lens[kk]]
                    lines[i] = L.topk_compress(row, k)
                    greedy[i] = vocab.decode(L.greedy_ids(row))
        out[key] = dict(itos=vocab.itos, sep="" if vocab.no_space else " ", lines=lines, greedy=greedy, by_patch=bp,
                        tags=list(tags), H=H, kind=vocab.kind)
        log(f"  log-probs ready for {key} ({'+'.join(tags)}): {len(lines):,} lines, alphabet {len(vocab.itos)}")
        del ms
    return out


_W = {}


def _winit(pkl, texts, orders):
    _W["D"] = pickle.load(open(pkl, "rb"))
    _W["lm"] = {n: L.CharLM(texts, n) for n in orders}


def _wtask(t):
    key, a, b, n, patches = t
    d, lm = _W["D"][key], _W["lm"][n]
    res = {}
    for p in patches:
        parts = [L.beam_decode(*d["lines"][i], d["itos"], lm, a, b).strip() for i in d["by_patch"].get(p, [])]
        res[p] = d["sep"].join(x for x in parts if x)
    return key, a, b, n, res


def greedy_texts(D, key, patches):
    d = D[key]
    return {p: d["sep"].join(t for t in (d["greedy"][i].strip() for i in d["by_patch"].get(p, [])) if t) for p in patches}


def run_lm(D, keys, cfgs, patches, texts, work, procs, log, chunk=8):
    """Decode `patches` for every (key, alpha, beta, order) in cfgs[key], in parallel. Returns {(key,a,b,n): {p: text}}."""
    from multiprocessing import get_context
    pkl = os.path.join(work, "lm_inputs.pkl")
    pickle.dump({k: D[k] for k in keys}, open(pkl, "wb"))
    orders = sorted({c[2] for k in keys for c in cfgs[k]})
    tasks = [(k, a, b, n, patches[s:s + chunk]) for k in keys for (a, b, n) in cfgs[k] for s in range(0, len(patches), chunk)]
    out = {}
    ctx = get_context("spawn")              # never fork a process that holds a CUDA context
    with ctx.Pool(procs, initializer=_winit, initargs=(pkl, texts, orders)) as pool:
        for k, a, b, n, res in pool.imap_unordered(_wtask, tasks):
            out.setdefault((k, a, b, n), {}).update(res)
    return out


def score(refs, hyps, n_boot=0):
    m = L.metrics(refs, hyps)
    if n_boot:
        import editdistance
        u = lambda s: s.replace("<FRAC>", "")
        e = np.array([editdistance.eval(u(r), u(h)) for r, h in zip(refs, hyps)], float)
        n = np.array([max(1, len(u(r))) for r in refs], float)
        idx = np.random.default_rng(0).integers(0, len(refs), (n_boot, len(refs)))
        bs = 100 * e[idx].sum(1) / n[idx].sum(1)
        m["CER_CI95"] = (round(float(np.percentile(bs, 2.5)), 2), round(float(np.percentile(bs, 97.5)), 2))
    return m


# ======================================================================= the whole final phase
def final_phase(OUT, WORK, META, log, device="cuda", procs=4, time_budget_s=900, ensemble_k=3, quick=False):
    import pandas as pd
    t0 = time.time()
    runs = os.path.join(OUT, "runs")
    R = load_results(runs)
    TEXT = META["text"]; VS, VR, TE = META["val_sel"], META["val_rep"], META["test"]
    texts = META["train_texts"]
    report = dict(results=R)
    okR = {t: r for t, r in R.items() if r.get("status") == "ok" and "val_sel" in r}
    log(f"FINAL PHASE: {len(okR)} finished jobs with results ({len(R) - len(okR)} failed or without evaluation)")

    # ---- 1. summary table
    rows = []
    for t, r in sorted(R.items()):
        sp = r.get("spec", {})
        row = dict(tag=t, status=r.get("status"), capped=r.get("capped"), priority=sp.get("priority"), hyp=",".join(r.get("hyp", [])),
                   desc=r.get("desc"), model=sp.get("size"), H=sp.get("H"), targets=sp.get("vocab") + ("/no-space" if sp.get("no_space") else ""),
                   params_M=r.get("params_M"), minutes=r.get("job_minutes"))
        if r.get("pre"):
            row.update(pre_lines_seen=r["pre"]["lines_seen"], pre_steps=f"{r['pre']['steps_done']}/{r['pre']['steps_planned']}")
        if r.get("ft"):
            row.update(ft_epochs=f"{r['ft']['epochs_done']}/{r['ft']['epochs_planned']}", best_epoch=r["ft"]["best_epoch"])
        for w in ("val_sel", "val_rep", "train150"):
            if w in r:
                row[f"{w}_CER"] = round(r[w]["CER"], 2); row[f"{w}_CER_nospace"] = round(r[w]["CER_nospace"], 2)
        if "val_rep" in r:
            row.update(val_rep_BLEU=round(r["val_rep"]["BLEU"], 2), val_rep_chrF=round(r["val_rep"]["chrF_pp"], 1),
                       val_rep_below50=f"{r['val_rep']['patches_below_50']}/{r['val_rep']['n']}")
            cg = content_gap(runs, t)
            if cg:
                r["content_gap"] = cg
                row.update(reads_gap=cg["gap"], own_patch_better_pct=cg["own_better_pct"])
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(OUT, "summary.csv"), index=False)
    with pd.option_context("display.max_columns", 50, "display.width", 250, "display.max_colwidth", 60):
        log(summary.drop(columns=["desc"]).to_string(index=False))

    # ---- 2. verdicts
    V, noise = verdicts(R, runs)
    report.update(verdicts=V, noise=noise)
    log("\nHYPOTHESIS VERDICTS (VAL-REP, never used for any choice; paired bootstrap over the same patches)")
    for v in V:
        log(f"  {v['id']:10s} {v['verdict']:28s} {v['variant']} vs {v['control']}: {v['detail']}")
    log(f"  run-to-run noise of the anchor (3 fine-tune seeds, VAL-REP CER): {noise['seeds']} -> sd {noise['sd']}")

    # ---- 3. which models get LM decoding
    cand = {t: r for t, r in okR.items() if r.get("ft") and not r["spec"].get("no_space")}
    groups = {}
    if "FT_A0_s0" in cand:
        groups["anchor"] = ["FT_A0_s0"]
    best = min(cand, key=lambda t: cand[t]["val_sel"]["primary"]) if cand else None
    if best:
        groups["best_single"] = [best]
    for t in ("COMBO", "FINAL_s0", "FINAL_s1"):
        if t in cand and t != best:
            groups[t.lower()] = [t]
    if best:
        fam = lambda t: (cand[t]["spec"]["H"], cand[t]["spec"]["vocab"])
        same = sorted((t for t in cand if fam(t) == fam(best)), key=lambda t: cand[t]["val_sel"]["primary"])[:ensemble_k]
        if len(same) >= 2:
            groups["ensemble"] = same
    log(f"\nLM decoding for: { {k: v for k, v in groups.items()} } (best single by VAL-SEL: {best})")
    if not groups:
        log("nothing to decode"); return report
    D = lm_inputs(runs, WORK, META, groups, set(VS) | set(VR) | set(TE), device, log)

    # ---- 4. LM grid on VAL-SEL (N3), in parallel on the CPUs
    keys = list(groups)
    if quick:
        grid1 = [(0.3, 1.0, 4), (0.3, 2.0, 4)]
    else:
        # beta (length bonus): 0.5 lost clearly on both CPU models (X6, X6b); 1.5 was best on X3a, 2.0 close
        grid1 = [(a, b, 4) for a in (0.2, 0.3, 0.45) for b in (1.0, 1.5, 2.0, 2.5)]
    lm4 = L.CharLM(texts, 4); d0 = D[keys[0]]; probe = VS[:6]
    t1 = time.time()
    for p in probe:
        for i in d0["by_patch"].get(p, []):
            L.beam_decode(*d0["lines"][i], d0["itos"], lm4, 0.3, 1.0)
    per_patch = (time.time() - t1) / max(1, len(probe))
    est = per_patch * len(VS) * len(keys) * (len(grid1) + 2) / max(1, procs) + 60
    left = time_budget_s - (time.time() - t0)
    log(f"  LM decoding speed: ~{per_patch:.2f}s per patch per setting on one core; full grid would take ~{L.fmt_s(est)}, "
        f"{L.fmt_s(left)} of the final budget left")
    if est > 0.6 * left:
        grid1 = [(0.3, b, 4) for b in (1.0, 1.5, 2.0)]
        keys = [k for k in keys if k in ("anchor", "best_single", "ensemble")]
        log(f"  -> reduced grid {grid1} for {keys}")
    res = run_lm(D, keys, {k: grid1 for k in keys}, VS, texts, WORK, procs, log)
    best_cfg, lm_table = {}, []
    refs_vs = [TEXT[p] for p in VS]
    for k in keys:
        g = score(refs_vs, [greedy_texts(D, k, VS)[p] for p in VS])["CER"]
        lm_table.append(dict(model=k, alpha="greedy", beta="", order="", val_sel_CER=round(g, 2)))
        scores = {}
        for (a, b, n) in grid1:
            c = score(refs_vs, [res[(k, a, b, n)][p] for p in VS])["CER"]
            scores[(a, b, n)] = c
            lm_table.append(dict(model=k, alpha=a, beta=b, order=n, val_sel_CER=round(c, 2)))
        best_cfg[k] = min(scores, key=scores.get)
    # orders 5 and 6 at each model's best alpha/beta
    if not quick and time_budget_s - (time.time() - t0) > 3 * per_patch * len(VS) * len(keys) * 2 / max(1, procs):
        cf = {k: [(best_cfg[k][0], best_cfg[k][1], n) for n in (5, 6)] for k in keys}
        res2 = run_lm(D, keys, cf, VS, texts, WORK, procs, log)
        for k in keys:
            for c in cf[k]:
                v = score(refs_vs, [res2[(k,) + c][p] for p in VS])["CER"]
                lm_table.append(dict(model=k, alpha=c[0], beta=c[1], order=c[2], val_sel_CER=round(v, 2)))
                cur = [x["val_sel_CER"] for x in lm_table if x["model"] == k and (x["alpha"], x["beta"], x["order"]) == best_cfg[k]][0]
                if v < cur:
                    best_cfg[k] = c
    lm_df = pd.DataFrame(lm_table); lm_df.to_csv(os.path.join(OUT, "lm_grid_val_sel.csv"), index=False)
    log("\nLM GRID on VAL-SEL (N3):")
    for k in keys:
        sub = lm_df[lm_df.model == k]
        log(f"  {k:12s} greedy {sub[sub.alpha == 'greedy'].val_sel_CER.iloc[0]:.2f} | best alpha/beta/order {best_cfg[k]} -> "
            f"{min(sub[sub.alpha != 'greedy'].val_sel_CER):.2f} | all: " +
            ", ".join(f"({r.alpha},{r.beta},{r.order}) {r.val_sel_CER}" for r in sub[sub.alpha != 'greedy'].itertuples()))

    # ---- 5. VAL-REP with the chosen settings
    rep = run_lm(D, keys, {k: [best_cfg[k]] for k in keys}, VR, texts, WORK, procs, log)
    refs_vr = [TEXT[p] for p in VR]
    pipe = []
    for k in keys:
        gs = score(refs_vr, [greedy_texts(D, k, VR)[p] for p in VR])
        ls = score(refs_vr, [rep[(k,) + best_cfg[k]][p] for p in VR])
        vs_lm = min(x["val_sel_CER"] for x in lm_table if x["model"] == k and x["alpha"] != "greedy")
        pipe.append(dict(pipeline=k, models="+".join(groups[k]), val_sel_LM_CER=round(vs_lm, 2), lm=best_cfg[k],
                         val_rep_greedy_CER=round(gs["CER"], 2), val_rep_LM_CER=round(ls["CER"], 2),
                         val_rep_LM_CER_nospace=round(ls["CER_nospace"], 2), val_rep_LM_BLEU=round(ls["BLEU"], 2),
                         val_rep_LM_chrF=round(ls["chrF_pp"], 1)))
    pipe_df = pd.DataFrame(pipe); pipe_df.to_csv(os.path.join(OUT, "pipelines_val.csv"), index=False)
    log("\nPIPELINES (greedy vs character-LM beam search):")
    log(pipe_df.to_string(index=False))

    # ---- 6. N5: put spaces back into the no-space model's reading
    ns_res = None
    if "NS" in okR:
        try:
            lm5 = L.CharLM(texts, 5)
            dsel, drep = preds(runs, "NS", "val_sel"), preds(runs, "NS", "val_rep")
            best_g, best_c = None, 1e9
            for g in (-3.0, -2.0, -1.0, 0.0, 1.0):
                c = score(list(dsel.ref), [L.restore_spaces(h, lm5, g) for h in dsel.hyp])["CER"]
                log(f"  NS space restoration gamma {g:+.1f}: VAL-SEL CER {c:.2f}")
                if c < best_c:
                    best_g, best_c = g, c
            rr = score(list(drep.ref), [L.restore_spaces(h, lm5, best_g) for h in drep.hyp])
            ns_res = dict(gamma=best_g, val_sel_CER=round(best_c, 2), val_rep_CER=round(rr["CER"], 2),
                          val_rep_CER_nospace=round(rr["CER_nospace"], 2))
            log(f"  N5: no-space model + LM space restoration -> VAL-REP CER {rr['CER']:.2f} (no-space {rr['CER_nospace']:.2f}); "
                f"anchor greedy VAL-REP CER {okR['FT_A0_s0']['val_rep']['CER'] if 'FT_A0_s0' in okR else 'n/a'}")
        except Exception as e:
            log(f"  NS space restoration failed: {e}")
    report["ns_restoration"] = ns_res

    # ---- 7. TEST, once, for the two pre-registered pipelines
    final_key = min(pipe, key=lambda r: r["val_sel_LM_CER"])["pipeline"]
    test_keys = ["anchor"] if "anchor" in keys else []
    if final_key not in test_keys:
        test_keys.append(final_key)
    tr = run_lm(D, test_keys, {k: [best_cfg[k]] for k in test_keys}, TE, texts, WORK, procs, log)
    refs_te = [TEXT[p] for p in TE]
    test = {}
    for k in test_keys:
        g_h = [greedy_texts(D, k, TE)[p] for p in TE]; l_h = [tr[(k,) + best_cfg[k]][p] for p in TE]
        test[k] = dict(models="+".join(groups[k]), lm=best_cfg[k], greedy=score(refs_te, g_h, 10000), lm_beam=score(refs_te, l_h, 10000))
        pd.DataFrame(dict(patch=TE, reference=refs_te, greedy=g_h, lm_beam=l_h)).to_csv(
            os.path.join(OUT, f"test_predictions_{k}.csv"), index=False, encoding="utf-8")
    report.update(test=test, final_pipeline=final_key, lm_best=best_cfg, pipelines=pipe)
    log(f"\nTEST ({len(TE)} patches, scored once; final pipeline chosen on VAL-SEL = '{final_key}'):")
    for k, v in test.items():
        for dec in ("greedy", "lm_beam"):
            m = v[dec]
            log(f"  {k:12s} {dec:8s} CER {m['CER']:6.2f} (95% CI {m['CER_CI95'][0]}-{m['CER_CI95'][1]}) | no-space {m['CER_nospace']:6.2f} | "
                f"WER {m['WER']:6.2f} | BLEU {m['BLEU']:6.2f} | chrF++ {m['chrF_pp']:5.1f} | <50%: {m['patches_below_50']}/{m['n']}")
    json.dump(report, open(os.path.join(OUT, "final_report.json"), "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=str)
    log(f"final phase took {L.fmt_s(time.time() - t0)}")
    return report


# ======================================================================= figures and REPORT.md
def figures(OUT, R, log):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    runs = os.path.join(OUT, "runs"); fig_dir = os.path.join(OUT, "figures"); os.makedirs(fig_dir, exist_ok=True)
    fig, ax = plt.subplots(1, 2, figsize=(18, 6))
    for t, r in sorted(R.items()):
        p = os.path.join(runs, t, "history.csv")
        if not os.path.exists(p):
            continue
        h = pd.read_csv(p)
        f = h[h.stage == "ft"]
        if len(f):
            ax[0].plot(f.step, f.primary, label=t, lw=2.2 if t == "FT_A0_s0" else 1.1)
        pr = h[h.stage == "pre"]
        if len(pr):
            ax[1].plot(pr.samples, pr.CER, marker="o", label=t)
    ax[0].axhline(84, color="r", ls=":", label="no-reading baseline ~84"); ax[0].set_ylim(0, 100)
    ax[0].set_title("fine-tuning: val-sel CER per epoch (no-space CER for NS)"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7, ncol=2)
    ax[1].set_title("pretraining: zero-shot CER on REAL val-sel vs synthetic lines seen"); ax[1].set_xlabel("synthetic lines seen")
    ax[1].legend(fontsize=7); ax[1].set_ylim(0, 105)
    fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "curves.png"), dpi=110); plt.close(fig)
    ok = [(t, r) for t, r in R.items() if r.get("status") == "ok" and "val_rep" in r]
    if ok:
        ok.sort(key=lambda x: x[1]["val_rep"]["CER"])
        fig, ax = plt.subplots(figsize=(12, 0.42 * len(ok) + 1.5))
        ax.barh([t for t, _ in ok], [r["val_rep"]["CER"] for _, r in ok], color=["tab:red" if t == "FT_A0_s0" else "tab:blue" for t, _ in ok])
        for k, (t, r) in enumerate(ok):
            ax.text(r["val_rep"]["CER"] + 0.3, k, f"{r['val_rep']['CER']:.1f}", va="center", fontsize=8)
        ax.invert_yaxis(); ax.set_xlabel("VAL-REP CER (greedy, lower is better); red = anchor")
        fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "val_rep_cer.png"), dpi=110); plt.close(fig)
    log(f"figures written to {fig_dir}")


def write_report(OUT, rep, run_info, log):
    R = rep.get("results", {})
    L_ = []
    w = L_.append
    w("# Notebook 05 - round-2 hypotheses at GPU scale: results\n")
    w(f"*Generated automatically at the end of the run. {run_info}*\n")
    w("Rules fixed before the run: VAL-SEL (first 150 validation patches) chose every epoch, LM weight and the final pipeline; "
      "VAL-REP (the rest of validation) is only reported; TEST was scored once, for the anchor and the pipeline VAL-SEL chose.\n")
    if rep.get("test"):
        w("## Headline: TEST\n")
        w("| pipeline | models | decoding | CER (95% CI) | CER no-space | WER | BLEU | chrF++ | patches <50% CER |")
        w("|---|---|---|---|---|---|---|---|---|")
        for k, v in rep["test"].items():
            for dec in ("greedy", "lm_beam"):
                m = v[dec]
                w(f"| {k}{' (FINAL)' if k == rep.get('final_pipeline') else ''} | {v['models']} | {dec}{' ' + str(v['lm']) if dec == 'lm_beam' else ''} | "
                  f"{m['CER']:.2f} ({m['CER_CI95'][0]}-{m['CER_CI95'][1]}) | {m['CER_nospace']:.2f} | {m['WER']:.2f} | {m['BLEU']:.2f} | "
                  f"{m['chrF_pp']:.1f} | {m['patches_below_50']}/{m['n']} |")
        w("\nFor reference, on this same TEST set: notebook 03 CER 73.1; SPAN v1 74.4; SPAN v2 69.9 (IAM-initialised; 72.6 from "
          "scratch); the CPU line baseline (X3a + character LM) **36.3** (44.9 greedy).\n")
    w("## Hypothesis verdicts (VAL-REP)\n")
    w("A change counts as BETTER/WORSE only if the paired-bootstrap 95% CI excludes zero AND the difference is larger than "
      "twice the anchor's seed-to-seed standard deviation.\n")
    w("| id | question | variant vs control | metric | variant | control | difference (95% CI) | verdict |")
    w("|---|---|---|---|---|---|---|---|")
    for v in rep.get("verdicts", []):
        if "delta" in v:
            w(f"| {v['id']} | {v['question']} | {v['variant']} vs {v['control']} | {v['metric']} | {v['variant_value']} | "
              f"{v['control_value']} | {v['delta']:+.2f} ({v['ci95'][0]:+.2f} to {v['ci95'][1]:+.2f}) | **{v['verdict']}** |")
        else:
            w(f"| {v['id']} | {v['question']} | {v['variant']} vs {v['control']} | {v['metric']} | | | | {v['verdict']}: {v['detail']} |")
    nz = rep.get("noise", {})
    w(f"\nRun-to-run noise: anchor VAL-REP CER over fine-tune seeds = {nz.get('seeds')} (sd {nz.get('sd')}).\n")
    if rep.get("pipelines"):
        w("## Decoding (N3) and ensembles (N15), VAL-REP\n")
        w("| pipeline | models | LM (alpha, beta, order) chosen on VAL-SEL | greedy CER | LM CER | LM CER no-space | LM BLEU |")
        w("|---|---|---|---|---|---|---|")
        for p in rep["pipelines"]:
            w(f"| {p['pipeline']} | {p['models']} | {p['lm']} | {p['val_rep_greedy_CER']} | {p['val_rep_LM_CER']} | "
              f"{p['val_rep_LM_CER_nospace']} | {p['val_rep_LM_BLEU']} |")
    if rep.get("ns_restoration"):
        n = rep["ns_restoration"]
        w(f"\n**N5 space restoration:** no-space model + LM spaces (gamma {n['gamma']}): VAL-REP CER {n['val_rep_CER']} "
          f"(no-space {n['val_rep_CER_nospace']}).\n")
    w("## Every job\n")
    w("*Reads?* = CER against each patch's own transcription vs against a patch with the same amount of text (VAL-REP). "
      "A reader is far better on its own patch (CPU line reader: own patch better on 149/150); a model that only learned "
      "how much text a page holds is a coin flip (notebook 03: 153/306; SPAN v1: 174/306).\n")
    w("| job | status | what changed | val-sel CER | val-rep CER | val-rep no-space | train-150 CER | reads? own vs same-length (own better) | epochs (best) | minutes |")
    w("|---|---|---|---|---|---|---|---|---|---|")
    for t, r in sorted(R.items(), key=lambda x: (x[1].get("val_rep", {}).get("CER", 999))):
        ft = r.get("ft") or {}
        cg = r.get("content_gap")
        reads = f"{cg['own']} vs {cg['neighbour']} ({cg['own_better_pct']}%)" if cg else "-"
        w(f"| {t} | {r.get('status')}{' (time-capped)' if r.get('capped') else ''} | {r.get('desc', '')} | "
          f"{r.get('val_sel', {}).get('CER', float('nan')):.2f} | {r.get('val_rep', {}).get('CER', float('nan')):.2f} | "
          f"{r.get('val_rep', {}).get('CER_nospace', float('nan')):.2f} | {r.get('train150', {}).get('CER', float('nan')):.2f} | "
          f"{reads} | {ft.get('epochs_done', '-')} ({ft.get('best_epoch', '-')}) | {r.get('job_minutes', '-')} |")
    a = R.get("FT_A0_s0", {})
    if a.get("n1n2"):
        w("\n## Error anatomy of the anchor (N1, N2)\n")
        an = a.get("anatomy", {})
        w(f"Errors on all validation: deletions {an.get('deletions_pct')}%, insertions {an.get('insertions_pct')}%, "
          f"substitutions {an.get('substitutions_pct')}%. Most missed: {an.get('most_missed')}. Top confusions: {an.get('top_confusions', [])[:8]}.\n")
        w("| patch feature | Spearman with CER | Spearman with deletion rate | CER by quartile (low -> high) |")
        w("|---|---|---|---|")
        for f, v in a["n1n2"].items():
            if isinstance(v, dict):
                w(f"| {f} | {v['spearman_cer']:+.2f} | {v['spearman_del']:+.2f} | {v['cer_by_quartile']} |")
    w("\n## Files\n`summary.csv` (every job), `runs/<job>/` (log.txt, history.csv, result.json, predictions, best.pt), "
      "`lm_grid_val_sel.csv`, `pipelines_val.csv`, `test_predictions_*.csv`, `figures/`, `logs.txt` (everything printed).\n")
    open(os.path.join(OUT, "REPORT.md"), "w", encoding="utf-8").write("\n".join(L_))
    log(f"wrote {os.path.join(OUT, 'REPORT.md')}")

In [ ]:
# Cell 10a - Noto Sans Modi (SIL Open Font License 1.1, notofonts.github.io), embedded so no extra input is needed
FONT_MD5 = '765e658a4554087f4c3af36edc18a6fa'
FONT_B64 = '...'

In [ ]:
# Cell 10 - the font, a check that Modi text is shaped correctly, and the synthetic-line renderer (background)
sys.path.insert(0, CODE)
import lr05_lib as L, lr05_run as LR, lr05_final as LF
FONT = f'{WORK}/NotoSansModi-Regular.ttf'
open(FONT, 'wb').write(base64.b64decode(FONT_B64))
assert hashlib.md5(open(FONT, 'rb').read()).hexdigest() == FONT_MD5, 'font file corrupted'

def raqm_ok():
    r = subprocess.run([sys.executable, '-c', "from PIL import features; print(features.check('raqm'))"],
                       capture_output=True, text=True)
    return r.stdout.strip() == 'True'

RAQM = raqm_ok()
if not RAQM and not LOCAL:
    log('Pillow cannot shape complex scripts here (raqm/fribidi missing). Trying to install them ...')
    subprocess.run('apt-get -qq update && apt-get -qq install -y libraqm0 libfribidi0 libharfbuzz0b',
                   shell=True, timeout=400, capture_output=True)
    RAQM = raqm_ok()
    if not RAQM:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '--upgrade', 'pillow'], timeout=400, capture_output=True)
        RAQM = raqm_ok()
log(f'complex-script shaping (raqm) for the renderer: {"OK" if RAQM else "NOT AVAILABLE"}')
if not RAQM:
    log('WARNING: synthetic Modi will be drawn WITHOUT shaping (vowel signs may sit in the wrong place). Every synthetic '
        'result is still produced, but read the synthetic-data comparisons with that in mind.')

# Experiments compare equal-size subsets (A_small/A_mid/A_big, B_mid, C_mid); only the FINAL models use a whole pool.
SIZES = (dict(A=600, A_small=60, A_mid=300, A_big=450, A_h64=300, B=400, B_mid=300, C=400, C_mid=300) if LOCAL else
         dict(A=200000, A_small=12000, A_mid=60000, A_big=120000, A_h64=60000, B=120000, B_mid=60000, C=120000, C_mid=60000))
_free_gb = shutil.disk_usage(WORK).free / 2**30
if not LOCAL and _free_gb < 15:      # the full pools need ~9 GB of scratch disk; fall back rather than fail
    SIZES.update(A=120000, B=60000, C=60000)
    log(f'NOTE: only {_free_gb:.0f} GB free scratch disk -> smaller synthetic pools (A 120k, B 60k, C 60k); experiments unchanged')
POOL_PLAN = [dict(name='A', n=SIZES['A'], style='plain', text='labels', seed=1,
                  milestones=sorted({SIZES['A_small'], SIZES['A_mid'], SIZES['A_big']}), h64_limit=SIZES['A_h64']),
             dict(name='B', n=SIZES['B'], style='realistic', text='labels', seed=2, milestones=[SIZES['B_mid']]),
             dict(name='C', n=SIZES['C'], style='plain', text='lm', seed=3, milestones=[SIZES['C_mid']])]
json.dump(POOL_PLAN, open(f'{WORK}/pool_plan.json', 'w'))
N_RENDER = max(1, min(3, (os.cpu_count() or 4) - 1))
RENDER_LOG = f'{OUT}/render_log.txt'
RENDERER = subprocess.Popen([sys.executable, '-u', f'{CODE}/render_pools.py', '--work', WORK, '--font', FONT,
                             '--plan', f'{WORK}/pool_plan.json', '--procs', str(N_RENDER)],
                            stdout=open(RENDER_LOG, 'w', encoding='utf-8'), stderr=subprocess.STDOUT,
                            env=dict(os.environ, PYTHONIOENCODING='utf-8'),
                            # lowest CPU priority: the GPU benchmark and the training jobs always get the CPU first
                            preexec_fn=(lambda: os.nice(19)) if hasattr(os, 'nice') else None)
log(f'[{clock()}] renderer started in the background with {N_RENDER} processes: '
    + ', '.join(f"pool {p['name']} = {p['n']:,} {p['style']} lines (text: {p['text']})" for p in POOL_PLAN))

# a look at what the renderer draws (saved for the record)
import matplotlib, warnings, logging
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', message='Glyph .* missing from')                  # Devanagari titles: no log flood
warnings.filterwarnings('ignore', message='Matplotlib currently does not support Devanagari')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
plt.rcParams['font.family'] = ['Nirmala UI', 'Noto Sans Devanagari', 'Lohit Devanagari', 'FreeSans', 'DejaVu Sans']
_rng = np.random.default_rng(5)
_chunks = L.label_chunks([L.normalise(t).replace('<FRAC>', '') for t in META_JSON['train_texts']], _rng)
fig, ax = plt.subplots(8, 1, figsize=(15, 8))
for k in range(8):
    style = 'realistic' if k % 2 else 'plain'
    lab = _chunks[int(_rng.integers(len(_chunks)))]
    o = L.render_line(lab, _rng, FONT, (40,), no_gaps=style == 'realistic', headline=style == 'realistic')
    if o is not None:
        ax[k].imshow(o[40], cmap='gray', vmin=0, vmax=255)
    ax[k].set_title(f'{style}: {lab}', fontsize=8, loc='left'); ax[k].axis('off')
fig.tight_layout(); fig.savefig(f'{FIGS}/synthetic_examples_main_process.png', dpi=80); plt.close(fig)

In [ ]:
# Cell 11 - real line images: each patch's own line pitch -> lines -> tight crop -> 40 px and 64 px versions,
# patch label split across its lines at word boundaries (re-aligned later by the EM jobs). Same code as the CPU work.
t0 = time.time()
W40, W64 = L.LineWriter(f'{WORK}/real_h40', 40), L.LineWriter(f'{WORK}/real_h64', 64)
PINFO, n_done = {}, 0
for r in ds:
    n = r['filename']
    if n not in KEEP:
        continue
    g = np.array(r['image'].convert('L'))
    lines, st = L.patch_to_lines(g, text[n], (40, 64))
    PINFO[n] = st
    for x in lines:
        info = dict(patch=n, split=splits[n], idx=x['idx'], n_lines=x['n_lines'])
        W40.add(x['imgs'][40], x['label'], info); W64.add(x['imgs'][64], x['label'], info)
    n_done += 1
    if n_done % 250 == 0:
        log(f'  [{clock()}] {n_done}/{len(KEEP)} patches -> {len(W40.widths):,} lines')
W40.close(); W64.close()
META_JSON['patch_info'] = PINFO
json.dump(META_JSON, open(f'{WORK}/meta.json', 'w', encoding='utf-8'), ensure_ascii=False)
REAL = L.LineStore(f'{WORK}/real_h40')
d = pd.DataFrame(REAL.info); d['w'] = REAL.widths; d['chars'] = [len(l) for l in REAL.labels]
N_TRAIN_LINES = int((d.split == 'train').sum())
log(f'[{clock()}] {len(REAL):,} line images from {len(PINFO):,} patches in {time.time() - t0:.0f}s'
    + ('' if LOCAL else f"  (the CPU-verified set has 5,901: {'MATCH' if len(REAL) == 5901 else 'DIFFERENT - check'})"))
log('lines per patch:', pd.Series([v['n_lines'] for v in PINFO.values()]).value_counts().sort_index().to_dict())
log('lines by split:', d.split.value_counts().to_dict())
log(f"width at 40 px: median {d.w.median():.0f}, 95% {d.w.quantile(.95):.0f} | characters per line: median {d.chars.median():.0f} | "
    f"CTC frames per character (stride 2): median {(d.w / 2 / d.chars.clip(lower=1)).median():.1f}")
pit = np.array([v['pitch'] for v in PINFO.values()])
log(f'line pitch per patch: 5% {np.percentile(pit, 5):.0f} | median {np.median(pit):.0f} | 95% {np.percentile(pit, 95):.0f} px')
ex = d[d.split == 'train'].sample(min(12, N_TRAIN_LINES), random_state=0).index
fig, ax = plt.subplots(len(ex), 1, figsize=(16, 1.1 * len(ex)))
for a, i in zip(np.atleast_1d(ax), ex):
    a.imshow(REAL.img(i), cmap='gray', vmin=0, vmax=255); a.axis('off')
    a.set_title(f"{REAL.info[i]['patch']} line {REAL.info[i]['idx'] + 1}/{REAL.info[i]['n_lines']} -> {REAL.labels[i]}", fontsize=8, loc='left')
fig.tight_layout(); fig.savefig(f'{FIGS}/real_line_examples.png', dpi=70); plt.close(fig)

In [ ]:
# Cell 12 - measure this GPU, then size every job so the whole plan fits the budget
bench_spec = dict(tag='BENCH', work=WORK, run_dir=f'{RUNS}/_bench', size='small', H=40, vocab='char', no_space=False,
                  deadline=DEADLINE_JOBS,
                  bench=[dict(size='small', H=40), dict(size='big', H=40), dict(size='small', H=64)])
if LOCAL:
    bench_spec.update(cpu=True, threads=2, bench=[dict(c, warm=1, steps=3) for c in bench_spec['bench']])
os.makedirs(bench_spec['run_dir'], exist_ok=True)
json.dump(bench_spec, open(f'{RUNS}/_bench/spec.json', 'w'))
rc = subprocess.run([sys.executable, '-u', f'{CODE}/job.py', '--spec', f'{RUNS}/_bench/spec.json']
                    + ([] if LOCAL else ['--gpu', str(GPUS[0])]), capture_output=True, text=True,
                    env=dict(os.environ, PYTHONIOENCODING='utf-8'))
for line in (rc.stdout + rc.stderr).splitlines():
    log('[bench]', line)
BENCH = json.load(open(f'{RUNS}/_bench/bench.json'))

BASE = (dict(pre_samples=320, ft_epochs=2, nopre_epochs=3, long_ft_epochs=4, long_pre_samples=640, batch=16) if LOCAL else
        dict(pre_samples=300000, ft_epochs=40, nopre_epochs=80, long_ft_epochs=120, long_pre_samples=900000, batch=32))
JOBS = LR.experiment_matrix(RUNS, BASE, SIZES)
avail = max(1, len(GPUS)) * (DEADLINE_JOBS - COMBO_RESERVE_MIN * 60 - time.time())
need = sum(LR.estimate_s(j, BENCH, N_TRAIN_LINES) for j in JOBS)          # every experiment, priority 1-3
if not LOCAL and 0.88 * avail < need:
    # quality first: rather than shrink the experiments, take up to one extra hour (Kaggle allows 12 h per session)
    extra = min(3600.0, (need / 0.88 - avail) / max(1, len(GPUS)))
    DEADLINE_JOBS += extra; DEADLINE_ALL += extra; avail += max(1, len(GPUS)) * extra
    log(f'GPU slower than planned -> budget extended by {extra / 60:.0f} min (to {(DEADLINE_ALL - T_START) / 3600:.1f} h) '
        f'to keep every experiment at full size')
# quality first: budgets only shrink if the GPU is much slower than expected, and GROW (up to 2x) if it is faster
SCALE = min(2.0, max(0.5, 0.88 * avail / need))
LR.apply_scale(JOBS, SCALE, BASE)
BASE_SCALED = dict(BASE, pre_samples=int(BASE['pre_samples'] * SCALE), ft_epochs=max(4 if not LOCAL else 1, int(round(BASE['ft_epochs'] * SCALE))))
log(f'[{clock()}] GPU throughput (training lines/s): ' + ', '.join(f"{k} {v['lines_per_s']:,.0f}" for k, v in BENCH.items()))
log(f'all experiments need ~{need / 60:.0f} GPU-min at base size; {avail / 60:.0f} GPU-min available before the final models '
    f'-> every budget x {SCALE:.2f} (all jobs scaled equally, so comparisons stay fair)')
log(f"{'job':10s} {'P':>2s} {'hypothesis':12s} {'pretrain lines':>15s} {'ft epochs':>9s} {'est. min':>8s}  what changes")
for j in sorted(JOBS, key=lambda j: j['priority']):
    log(f"{j['tag']:10s} {j['priority']:>2d} {','.join(j['hyp']):12s} {(j['pretrain'] or {}).get('samples', 0):>15,} "
        f"{(j['finetune'] or {}).get('epochs', 0):>9} {LR.estimate_s(j, BENCH, N_TRAIN_LINES) / 60:8.1f}  {j['desc']}")

In [ ]:
# Cell 13 - THE RUN: every job, two at a time (one per GPU), highest priority first, streamed below as [job] lines.
# Each job logs its loss, speed, blank share, predicted length, and after every epoch the validation CER, a sample
# reading and the error mix. A dashboard is printed every 5 minutes. Nothing here needs watching.
log(f'[{clock()}] starting the scheduler; jobs must finish by {time.strftime("%H:%M:%S", time.localtime(DEADLINE_JOBS))} '
    f'(local time of this machine); priority-3 jobs only start if they leave {COMBO_RESERVE_MIN} min for the combination and FINAL models')
SCHED = LR.Scheduler(JOBS, GPUS or [0, 1], f'{CODE}/job.py', WORK, RUNS, DEADLINE_JOBS, BENCH, N_TRAIN_LINES, log,
                     poll=2 if LOCAL else 10, renderer=RENDERER, reserve={3: COMBO_RESERVE_MIN * 60},
                     cpu=LOCAL, threads=2, tails=[(RENDER_LOG, 'render')])
STATE = SCHED.run()
log(f'[{clock()}] main phase finished:', {s: [t for t, v in STATE.items() if v == s] for s in set(STATE.values())})
for t, why in SCHED.notes.items():
    log(f'   {t}: {why}')

In [ ]:
# Cell 14 - the COMBINATION and the FINAL models, built automatically from the results above.
#   COMBO      = every change that beat the anchor on VAL-SEL by more than the noise margin, together, at the SAME length
#                as the experiments (answers: do the winners add up?)
#   FINAL_s0/1 = the same recipe (the anchor recipe if nothing won) on the WHOLE synthetic pool of the winning style,
#                trained as long as the remaining time allows, two seeds - the final model (the two are also ensembled)
R = LF.load_results(RUNS)
sel = [R[t]['val_sel']['primary'] for t in ('FT_A0_s0', 'FT_A0_s1', 'FT_A0_s2') if t in R and R[t].get('status') == 'ok']
MARGIN = max(1.0, 2 * float(np.std(sel, ddof=1))) if len(sel) >= 2 else 1.5
log(f'[{clock()}] anchor seeds on VAL-SEL: {[round(s, 2) for s in sel]} -> a change must win by more than {MARGIN:.2f} CER points')
CJOBS, WHY = LR.combo_jobs(R, BASE_SCALED, SIZES, RUNS, MARGIN, log)
for w in WHY:
    log('  ', w)
left = DEADLINE_JOBS - time.time()
combo_est = sum(LR.estimate_s(j, BENCH, N_TRAIN_LINES) * SCHED.factor() for j in CJOBS if j['tag'] == 'COMBO')
# two GPUs: COMBO then FINAL_s1 on one, FINAL_s0 on the other; one GPU: everything one after another
target = 0.88 * ((left - combo_est) if len(GPUS or [0, 1]) >= 2 else (left - combo_est) / 2)
for j in CJOBS:
    if j['tag'].startswith('FINAL'):
        LR.fill_to(j, target, BENCH, N_TRAIN_LINES, SCHED.factor())
        log(f"  {j['tag']}: {j['pretrain']['samples']:,} synthetic lines of pretraining, {j['finetune']['epochs']} real epochs "
            f"-> about {LR.estimate_s(j, BENCH, N_TRAIN_LINES) * SCHED.factor() / 60:.0f} min ({left / 60:.0f} min left before the final phase)")
if CJOBS and time.time() < DEADLINE_JOBS - 120:
    SCHED2 = LR.Scheduler(CJOBS, GPUS or [0, 1], f'{CODE}/job.py', WORK, RUNS, DEADLINE_JOBS, BENCH, N_TRAIN_LINES, log,
                          poll=2 if LOCAL else 10, renderer=RENDERER, cpu=LOCAL, threads=2, ratios=SCHED.ratios)
    log(f'[{clock()}] combination phase:', SCHED2.run())
    for t, why in SCHED2.notes.items():
        log(f'   {t}: {why}')
else:
    log('no time left for the combination' if CJOBS else 'no combination to run')

# any time still left goes to the priority-3 jobs that were skipped for lack of time
LEFT = [j for j in JOBS if STATE.get(j['tag']) == 'skipped' and 'time' in SCHED.notes.get(j['tag'], '')]
if LEFT and time.time() < DEADLINE_JOBS - 180:
    log(f'[{clock()}] leftover time {(DEADLINE_JOBS - time.time()) / 60:.1f} min -> retrying skipped jobs: {[j["tag"] for j in LEFT]}')
    SCHED3 = LR.Scheduler(LEFT, GPUS or [0, 1], f'{CODE}/job.py', WORK, RUNS, DEADLINE_JOBS, BENCH, N_TRAIN_LINES, log,
                          poll=2 if LOCAL else 10, renderer=RENDERER, cpu=LOCAL, threads=2, known=STATE, ratios=SCHED.ratios)
    log(f'[{clock()}] leftover phase:', SCHED3.run())

In [ ]:
# Cell 15 - FINAL PHASE: verdict for every hypothesis, character-LM decoding (N3), ensemble (N15), space restoration
# for the no-space model (N5), the TEST score (once), figures and REPORT.md
if RENDERER.poll() is None:
    RENDERER.terminate()
log(f'[{clock()}] final phase, {(DEADLINE_ALL - time.time()) / 60:.1f} min left in the budget')
dev = 'cuda' if (GPUS and not LOCAL and torch.cuda.is_available()) else 'cpu'
REP = LF.final_phase(OUT, WORK, META_JSON, log, device=dev, procs=max(1, (os.cpu_count() or 2)),
                     time_budget_s=max(180, DEADLINE_ALL - time.time() - 90), quick=LOCAL)
LF.figures(OUT, LF.load_results(RUNS), log)
RUN_INFO = (f"GPUs: {smi.replace(chr(10), '; ') or 'none'}; budget scale {SCALE:.2f}; raqm shaping {'on' if RAQM else 'OFF'}; "
            f"total time {(time.time() - T_START) / 60:.0f} min")
LF.write_report(OUT, REP, RUN_INFO, log)

In [ ]:
# Cell 16 - the report, and one zip with every small output file (download this one)
print(open(f'{OUT}/REPORT.md', encoding='utf-8').read())
with zipfile.ZipFile(f'{OUT}/results_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in ['REPORT.md', 'summary.csv', 'logs.txt', 'render_log.txt', 'lm_grid_val_sel.csv', 'pipelines_val.csv',
              'final_report.json'] + [os.path.relpath(f, OUT) for f in glob.glob(f'{OUT}/test_predictions_*.csv')]:
        if os.path.exists(f'{OUT}/{p}'):
            z.write(f'{OUT}/{p}', p)
    for f in glob.glob(f'{FIGS}/*.png') + [f for f in glob.glob(f'{RUNS}/*/*')
                                           if f.endswith(('.txt', '.csv', '.json', '.png')) and not f.endswith('spec.json')]:
        z.write(f, os.path.relpath(f, OUT))
log(f'[{clock()}] DONE. results_bundle.zip = {os.path.getsize(f"{OUT}/results_bundle.zip") / 2**20:.1f} MB | '
    f'total {(time.time() - T_START) / 60:.0f} min')

## Reading the output
* **`REPORT.md`** answers every question: the TEST headline, a verdict per hypothesis (BETTER / WORSE / INCONCLUSIVE, with
  the difference and its 95% CI), decoding and ensemble results, every job, and the error anatomy (N1/N2).
* **Per job** (`runs/<job>/log.txt`): pretraining logs every 100 steps (loss, learning rate, gradient norm, blank share,
  predicted vs true length, lines/s, memory) and the zero-shot CER on REAL validation lines 4 times; fine-tuning logs every
  epoch (validation CER, no-space CER, WER, BLEU, chrF++, patches below 50%, length ratio, blank share) and every 5 epochs a
  sample reading, the deletion/insertion/substitution mix and the recall of the small marks (space, ा, ्, र, ं, ि, ी, ु, ू, े, ो).
* **The final model** is `FINAL_s0` / `FINAL_s1` (or their ensemble): the winning recipe trained as long as the time
  allowed. REPORT.md's TEST table says which pipeline VAL-SEL picked and how it decodes; its checkpoints are
  `runs/FINAL_s*/best.pt`.
* **Reference points:** reading nothing scores ~84 CER; the letter-frequency plateau is a CTC loss of 3.32; on the same
  TEST set notebook 03 scored CER 73.1, SPAN v1 74.4, and the CPU baseline (X3a + character LM) **36.3**.
* Send back `results_bundle.zip` (or `REPORT.md` + `logs.txt`).